# MAVIS — Análisis de alineamiento multimodal para detección de desinformación
## Objetivo y encuadre del TFG
### Hipótesis transversal
### Los tres datasets — dificultad creciente
### Los cuatro modelos


## 1. Setup


In [ ]:
import sys
sys.path.insert(0, r"C:\Users\Usuario\Documents\Universidad\4year\Segundo Semestre\TFG\TFG REPOSITORIO\experiments\analysis")

import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import warnings
warnings.filterwarnings('ignore')

import mavis_analysis as ma

REPO = ma.REPO_ROOT
print("REPO:", REPO)
print("FIG_DIR:", ma.FIG_DIR)


In [ ]:
# Availability table — muestra qué (dataset, model, modality) está disponible
avail = ma.availability_table()
print("\n=== Tabla de disponibilidad de datos ===")
pd.set_option('display.max_columns', 20)
pd.set_option('display.width', 120)
display(avail)


### Interpretación de la tabla de disponibilidad


## Resumen ejecutivo


In [ ]:
# Executive summary — headline metrics per model
# Self-contained: recomputes the few needed numbers directly from DBs
import sys
sys.path.insert(0, r"C:\Users\Usuario\Documents\Universidad\4year\Segundo Semestre\TFG\TFG REPOSITORIO\experiments\analysis")
import json
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')
import mavis_analysis as ma

_rng_es = np.random.default_rng(42)

def _exec_summary_a1_auc(model, by_id_es):
    """AUC for M3A A1 (score = -cos(video, summary), real=0 vs MM-swap=1)."""
    info = ma.available('M3A', model)
    mods = info.get('modalities', {}) if info.get('exists') else {}
    if not info.get('exists') or 'video' not in mods or 'text_summary' not in mods:
        return np.nan
    try:
        conn = ma.connect(ma.db_path('M3A', model))
        V = ma.load_globals(conn, 'video')
        T = ma.load_globals(conn, 'text_summary')
        conn.close()
    except Exception:
        return np.nan
    scores_l, labels_l = [], []
    for rid in V:
        if rid not in T or rid not in by_id_es:
            continue
        src = by_id_es[rid].get('mm_sources', {}).get('text')
        if not src or src not in T:
            continue
        scores_l.append(-ma.cos(V[rid], T[rid])); labels_l.append(0)
        scores_l.append(-ma.cos(V[rid], T[src])); labels_l.append(1)
    if not scores_l:
        return np.nan
    try:
        from sklearn.metrics import roc_auc_score
        return float(roc_auc_score(np.array(labels_l), np.array(scores_l, float)))
    except Exception:
        return np.nan

def _exec_summary_fakevv_acc(model):
    """FakeVV E1 paired accuracy for a model."""
    info = ma.available('FakeVV', model)
    mods = info.get('modalities', {}) if info.get('exists') else {}
    if not (info.get('exists') and 'video' in mods and 'text_real' in mods and 'text_fake' in mods):
        return np.nan
    try:
        conn = ma.connect(ma.db_path('FakeVV', model))
        vid  = ma.load_globals(conn, 'video')
        tr   = ma.load_globals(conn, 'text_real')
        tf   = ma.load_globals(conn, 'text_fake')
        conn.close()
    except Exception:
        return np.nan
    ids = sorted(set(vid) & set(tr) & set(tf))
    if not ids:
        return np.nan
    return float(np.mean([ma.cos(vid[r], tr[r]) > ma.cos(vid[r], tf[r]) for r in ids]))

def _exec_summary_gl_e1_auc(model):
    """GroundLie360 E1 AUC(-cos(video,title)) for a model."""
    info = ma.available('GroundLie360', model)
    mods = info.get('modalities', {}) if info.get('exists') else {}
    if not info.get('exists') or 'video' not in mods or 'text_title' not in mods:
        return np.nan
    try:
        conn = ma.connect(ma.db_path('GroundLie360', model))
        lab  = ma.load_labels(conn)
        vid  = ma.load_globals(conn, 'video')
        tit  = ma.load_globals(conn, 'text_title')
        conn.close()
    except Exception:
        return np.nan
    ids = sorted(set(vid) & set(tit) & set(lab.index))
    if not ids:
        return np.nan
    scores = [-ma.cos(vid[r], tit[r]) for r in ids]
    labels = [int(lab.loc[r, 'binary_label']) for r in ids]
    return ma.auc_fake(np.array(scores), np.array(labels))

def _exec_summary_gl_e6b_auc(model):
    """GroundLie360 E6b AUC(perimeter) for a model."""
    info = ma.available('GroundLie360', model)
    mods = info.get('modalities', {}) if info.get('exists') else {}
    needed = ['text_title', 'video', 'transcript']
    if not info.get('exists') or any(m not in mods for m in needed):
        return np.nan
    try:
        conn = ma.connect(ma.db_path('GroundLie360', model))
        lab  = ma.load_labels(conn)
        Ti   = ma.load_globals(conn, 'text_title')
        Vi   = ma.load_globals(conn, 'video')
        Tr   = ma.load_globals(conn, 'transcript')
        conn.close()
    except Exception:
        return np.nan
    ids = sorted(set(Ti) & set(Vi) & set(Tr) & set(lab.index))
    if not ids:
        return np.nan
    perims = [ma.perimeter3(Ti[r], Vi[r], Tr[r]) for r in ids]
    labels = [int(lab.loc[r, 'binary_label']) for r in ids]
    return ma.auc_fake(np.array(perims), np.array(labels))

# Load M3A index once for exec summary
_M3A_INDEX_ES = ma.REPO_ROOT / "experiments/M3A/m3a_index_20000.json"
_by_id_es = {}
if _M3A_INDEX_ES.exists():
    _by_id_es = {e['raw_id']: e for e in json.load(open(_M3A_INDEX_ES, encoding='utf-8'))}

_all_models = ['qw2b', 'qw8b', 'wave', 'ge2']
_rows_es = []
for _m in _all_models:
    _rows_es.append({
        'Model':            ma.MODEL_DISPLAY[_m],
        'FakeVV E1 acc':   round(_exec_summary_fakevv_acc(_m),   3) if not np.isnan(_exec_summary_fakevv_acc(_m))   else float('nan'),
        'GL360 E1 AUC':    round(_exec_summary_gl_e1_auc(_m),    3) if not np.isnan(_exec_summary_gl_e1_auc(_m))    else float('nan'),
        'GL360 E6b AUC':   round(_exec_summary_gl_e6b_auc(_m),   3) if not np.isnan(_exec_summary_gl_e6b_auc(_m))   else float('nan'),
        'M3A A1 AUC':      round(_exec_summary_a1_auc(_m, _by_id_es), 3) if not np.isnan(_exec_summary_a1_auc(_m, _by_id_es)) else float('nan'),
    })
df_exec = pd.DataFrame(_rows_es).set_index('Model')
display(df_exec)

# Save as matplotlib table figure
with plt.rc_context(ma._RC):
    _fig_es, _ax_es = plt.subplots(figsize=(8, 2.2))
    _ax_es.axis('off')
    _col_labels = list(df_exec.columns)
    _row_labels  = list(df_exec.index)
    _cell_text   = [[f'{v:.3f}' if not (isinstance(v, float) and np.isnan(v)) else 'n/a'
                     for v in row] for row in df_exec.values]
    _tbl = _ax_es.table(
        cellText=_cell_text, rowLabels=_row_labels, colLabels=_col_labels,
        cellLoc='center', loc='center',
        bbox=[0, 0, 1, 1],
    )
    _tbl.auto_set_font_size(False)
    _tbl.set_fontsize(9)
    _ax_es.set_title('MAVIS — Executive Summary: headline metrics per model', pad=8, fontsize=11)
    plt.tight_layout()
    _fig_es.savefig(str(ma.FIG_DIR / 'exec_summary.png'), dpi=150, bbox_inches='tight')
    plt.show(); plt.close('all')
print("Saved exec_summary.png")


### Hallazgos clave


### Procedencia de datos y advertencias metodológicas


## 2. Caracterización de los datasets (EDA)


In [ ]:
# ── EDA: imports y helpers ────────────────────────────────────────────────────
import sys
sys.path.insert(0, r"C:\Users\Usuario\Documents\Universidad\4year\Segundo Semestre\TFG\TFG REPOSITORIO\experiments\analysis")
import json
import sqlite3
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from collections import Counter
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')
import mavis_analysis as ma

REPO = ma.REPO_ROOT

def _try_connect(dataset, model):
    """Return a sqlite3 connection or None if DB not found."""
    try:
        p = ma.db_path(dataset, model)
        if p.exists():
            return sqlite3.connect(str(p))
    except Exception:
        pass
    return None

def _prefer_conn(dataset, prefer_order):
    """Return (model, conn) for first available model in prefer_order."""
    for m in prefer_order:
        c = _try_connect(dataset, m)
        if c is not None:
            return m, c
    return None, None


### 2.1 Resumen comparativo de los tres datasets


In [ ]:
# ── Cross-dataset overview DataFrame ─────────────────────────────────────────
_ov_rows = []

# ---- GroundLie360 ----
_gl_model, _gl_conn = _prefer_conn('GroundLie360', ['qw2b', 'qw8b', 'ge2'])
if _gl_conn is not None:
    _gl_lab = pd.read_sql_query('SELECT * FROM groundlie_labels', _gl_conn).set_index('raw_id')
    _gl_N = len(_gl_lab)
    _gl_n_fake = int(_gl_lab.binary_label.sum())
    _gl_tables = [r[0] for r in _gl_conn.execute("SELECT name FROM sqlite_master WHERE type='table'").fetchall()]
    if 'video_metadata' in _gl_tables:
        _gl_dur = pd.read_sql_query('SELECT duration_seconds FROM video_metadata', _gl_conn)
        _gl_mean_dur = float(_gl_dur.duration_seconds.mean())
    else:
        _gl_mean_dur = float('nan')
    if 'transcripts' in _gl_tables:
        _gl_tr_ids = set(r[0] for r in _gl_conn.execute(
            "SELECT DISTINCT raw_id FROM transcripts WHERE model!='NO_AUDIO'").fetchall())
        _gl_tr_cov = len(_gl_tr_ids) / _gl_N if _gl_N > 0 else float('nan')
    else:
        _gl_tr_cov = float('nan')
    _gl_conn.close()
    _ov_rows.append({
        'Dataset': 'GroundLie360', 'N': _gl_N,
        '%_fake': round(100 * _gl_n_fake / _gl_N, 1),
        '%_transcript': round(100 * _gl_tr_cov, 1),
        'mean_dur_s': round(_gl_mean_dur, 1),
        'modalities': 'video, text_title, transcript',
        'multilingual': 'No',
    })
else:
    print("[skip] GroundLie360 overview: no DB available")

# ---- FakeVV ----
_fvv_model, _fvv_conn = _prefer_conn('FakeVV', ['qw2b', 'qw8b', 'ge2'])
_FAKEVV_INDEX = REPO / "experiments/FakeVV_testset/google_embeddings2/test_index.json"
_fvv_idx = json.load(open(_FAKEVV_INDEX, encoding='utf-8')) if _FAKEVV_INDEX.exists() else []
_fvv_N = len(_fvv_idx)
_fvv_mean_dur = float('nan')
if _fvv_conn is not None:
    _fvv_tables = [r[0] for r in _fvv_conn.execute("SELECT name FROM sqlite_master WHERE type='table'").fetchall()]
    if 'video_metadata' in _fvv_tables:
        _fvv_dur = pd.read_sql_query('SELECT duration_seconds FROM video_metadata', _fvv_conn)
        _fvv_mean_dur = float(_fvv_dur.duration_seconds.mean())
    if 'transcripts' in _fvv_tables:
        _fvv_tr_ids = set(r[0] for r in _fvv_conn.execute(
            "SELECT DISTINCT raw_id FROM transcripts").fetchall())
        _fvv_tr_cov = len(_fvv_tr_ids) / _fvv_N if _fvv_N > 0 else float('nan')
    else:
        _fvv_tr_cov = float('nan')
    _fvv_conn.close()
else:
    _fvv_tr_cov = float('nan')
_ov_rows.append({
    'Dataset': 'FakeVV', 'N': _fvv_N,
    '%_fake': 100.0,  # every clip has a fake title (paired)
    '%_transcript': round(100 * _fvv_tr_cov, 1) if not (isinstance(_fvv_tr_cov, float) and (_fvv_tr_cov != _fvv_tr_cov)) else float('nan'),
    'mean_dur_s': round(_fvv_mean_dur, 1),
    'modalities': 'video, text_real, text_fake, transcript',
    'multilingual': 'No',
})

# ---- M3A ----
_m3a_model, _m3a_conn = _prefer_conn('M3A', ['qw2b', 'qw8b', 'ge2'])
_M3A_INDEX = REPO / "experiments/M3A/m3a_index_20000.json"
_m3a_idx = json.load(open(_M3A_INDEX, encoding='utf-8')) if _M3A_INDEX.exists() else []
_m3a_N = len(_m3a_idx)
_m3a_mean_dur = float('nan')
_m3a_tr_cov = float('nan')
if _m3a_conn is not None:
    _m3a_tables = [r[0] for r in _m3a_conn.execute("SELECT name FROM sqlite_master WHERE type='table'").fetchall()]
    if 'video_metadata' in _m3a_tables:
        _m3a_dur = pd.read_sql_query('SELECT duration_seconds FROM video_metadata', _m3a_conn)
        _m3a_mean_dur = float(_m3a_dur.duration_seconds.mean())
    if 'transcripts' in _m3a_tables:
        _m3a_tr_ids = set(r[0] for r in _m3a_conn.execute(
            "SELECT DISTINCT raw_id FROM transcripts").fetchall())
        _m3a_tr_cov = len(_m3a_tr_ids) / _m3a_N if _m3a_N > 0 else float('nan')
    _m3a_conn.close()
_ov_rows.append({
    'Dataset': 'M3A', 'N': _m3a_N,
    '%_fake': 100.0,   # all have MM swaps in index; real=original pair
    '%_transcript': round(100 * _m3a_tr_cov, 1) if not (isinstance(_m3a_tr_cov, float) and (_m3a_tr_cov != _m3a_tr_cov)) else float('nan'),
    'mean_dur_s': round(_m3a_mean_dur, 1),
    'modalities': 'video, text_summary, transcript, (NEM variants)',
    'multilingual': 'Sí',
})

df_overview = pd.DataFrame(_ov_rows).set_index('Dataset')
print("=== Cross-dataset overview ===")
display(df_overview)


### 2.2 GroundLie360 — Co-ocurrencia de tipos de mentira


In [ ]:
# ── GroundLie360: lie-type co-occurrence heatmap ──────────────────────────────
_gl2_model, _gl2_conn = _prefer_conn('GroundLie360', ['qw2b', 'qw8b', 'ge2'])
_LIE_TYPES = ['false_title', 'false_speech', 'temporal_edit', 'cgi', 'contradictory', 'unsupported']

if _gl2_conn is not None:
    _gl2_lab = pd.read_sql_query('SELECT * FROM groundlie_labels', _gl2_conn).set_index('raw_id')
    _gl2_conn.close()

    # per-type counts
    _present = [c for c in _LIE_TYPES if c in _gl2_lab.columns]
    _type_counts = {c: int(_gl2_lab[c].sum()) for c in _present}
    print("Lie type counts:", _type_counts)

    # co-occurrence matrix
    _co = np.zeros((len(_present), len(_present)), dtype=int)
    for i, ci in enumerate(_present):
        for j, cj in enumerate(_present):
            if i == j:
                _co[i, j] = int(_gl2_lab[ci].sum())
            else:
                _co[i, j] = int((_gl2_lab[ci] & _gl2_lab[cj]).sum())

    _co_df = pd.DataFrame(_co, index=_present, columns=_present)
    print("\nCo-occurrence matrix:")
    display(_co_df)

    with plt.rc_context(ma._RC):
        _fig_co, _ax_co = plt.subplots(figsize=(7, 6))
        _im_co = _ax_co.imshow(_co, cmap='Blues', aspect='auto')
        _ax_co.set_xticks(range(len(_present)))
        _ax_co.set_yticks(range(len(_present)))
        _ax_co.set_xticklabels(_present, rotation=30, ha='right', fontsize=9)
        _ax_co.set_yticklabels(_present, fontsize=9)
        for i in range(len(_present)):
            for j in range(len(_present)):
                _ax_co.text(j, i, str(_co[i, j]), ha='center', va='center',
                            fontsize=9, color='white' if _co[i, j] > _co.max() * 0.6 else 'black')
        plt.colorbar(_im_co, ax=_ax_co, shrink=0.7, label='Co-occurrence count')
        _ax_co.set_title('GroundLie360 — Lie-type co-occurrence (6×6)', pad=10)
        plt.tight_layout()
        _fig_co.savefig(str(ma.FIG_DIR / 'eda_gl360_lietype_cooccurrence.png'), dpi=150, bbox_inches='tight')
        plt.show(); plt.close('all')
    print("Saved eda_gl360_lietype_cooccurrence.png")
else:
    print("[skip] GL360 co-occurrence: no DB available")


### 2.3 FakeVV — Distribución de categorías


In [ ]:
# ── FakeVV: category distribution ────────────────────────────────────────────
if _fvv_idx:
    _fvv_cats = Counter(e.get('category', 'unknown') for e in _fvv_idx)
    _cats = sorted(_fvv_cats.keys())
    _cnts = [_fvv_cats[c] for c in _cats]

    with plt.rc_context(ma._RC):
        _fig_fvv, _ax_fvv = plt.subplots(figsize=(6, 4))
        _bars_fvv = _ax_fvv.bar(_cats, _cnts, color='#4472C4', edgecolor='white', linewidth=0.7)
        for b, v in zip(_bars_fvv, _cnts):
            _ax_fvv.text(b.get_x() + b.get_width()/2, v + 3, str(v),
                         ha='center', va='bottom', fontsize=9)
        _ax_fvv.set_ylabel('Number of clips')
        _ax_fvv.set_xlabel('Entity category (swapped in title)')
        _ax_fvv.set_title('FakeVV — Category distribution (N=997)')
        plt.tight_layout()
        _fig_fvv.savefig(str(ma.FIG_DIR / 'eda_fakevv_categories.png'), dpi=150, bbox_inches='tight')
        plt.show(); plt.close('all')
    print("Saved eda_fakevv_categories.png")
    print("Category counts:", dict(_fvv_cats))
else:
    print("[skip] FakeVV category chart: no index loaded")


### 2.4 M3A — Distribución geográfica y temática


In [ ]:
# ── M3A: geography & topic distributions ─────────────────────────────────────
if _m3a_idx:
    _geos  = Counter(e.get('geography', 'unknown') for e in _m3a_idx)
    _topics = Counter(e.get('topic', 'unknown') for e in _m3a_idx)

    _geo_sorted   = sorted(_geos.items(),   key=lambda x: -x[1])
    _topic_sorted = sorted(_topics.items(), key=lambda x: -x[1])

    with plt.rc_context(ma._RC):
        _fig_m3a, (_ax_geo, _ax_top) = plt.subplots(1, 2, figsize=(13, 4))

        _g_labels, _g_vals = zip(*_geo_sorted)
        _bars_g = _ax_geo.bar(_g_labels, _g_vals, color='#70AD47', edgecolor='white', linewidth=0.7)
        for b, v in zip(_bars_g, _g_vals):
            _ax_geo.text(b.get_x() + b.get_width()/2, v + 80, str(v),
                         ha='center', va='bottom', fontsize=8)
        _ax_geo.set_ylabel('Video count')
        _ax_geo.set_xlabel('Geography')
        _ax_geo.set_title('M3A — Geography distribution')
        _ax_geo.tick_params(axis='x', rotation=25)

        _t_labels, _t_vals = zip(*_topic_sorted)
        _bars_t = _ax_top.bar(_t_labels, _t_vals, color='#ED7D31', edgecolor='white', linewidth=0.7)
        for b, v in zip(_bars_t, _t_vals):
            _ax_top.text(b.get_x() + b.get_width()/2, v + 30, str(v),
                         ha='center', va='bottom', fontsize=7)
        _ax_top.set_ylabel('Video count')
        _ax_top.set_xlabel('Topic')
        _ax_top.set_title('M3A — Topic distribution')
        _ax_top.tick_params(axis='x', rotation=35)

        plt.tight_layout()
        _fig_m3a.savefig(str(ma.FIG_DIR / 'eda_m3a_strata.png'), dpi=150, bbox_inches='tight')
        plt.show(); plt.close('all')
    print("Saved eda_m3a_strata.png")
else:
    print("[skip] M3A strata chart: no index loaded")


### 2.5 Distribución de duración de vídeo


In [ ]:
# ── Duration overlay across 3 datasets ───────────────────────────────────────
_dur_datasets = {}

# GroundLie360
_gl3_model, _gl3_conn = _prefer_conn('GroundLie360', ['qw2b', 'qw8b', 'ge2'])
if _gl3_conn is not None:
    _gl3_tabs = [r[0] for r in _gl3_conn.execute("SELECT name FROM sqlite_master WHERE type='table'").fetchall()]
    if 'video_metadata' in _gl3_tabs:
        _gl3_dur = pd.read_sql_query('SELECT duration_seconds FROM video_metadata', _gl3_conn)
        _dur_datasets['GroundLie360'] = _gl3_dur.duration_seconds.dropna().values
    _gl3_conn.close()

# FakeVV
_fvv3_model, _fvv3_conn = _prefer_conn('FakeVV', ['qw2b', 'qw8b', 'ge2'])
if _fvv3_conn is not None:
    _fvv3_tabs = [r[0] for r in _fvv3_conn.execute("SELECT name FROM sqlite_master WHERE type='table'").fetchall()]
    if 'video_metadata' in _fvv3_tabs:
        _fvv3_dur = pd.read_sql_query('SELECT duration_seconds FROM video_metadata', _fvv3_conn)
        _dur_datasets['FakeVV'] = _fvv3_dur.duration_seconds.dropna().values
    _fvv3_conn.close()

# M3A
_m3a3_model, _m3a3_conn = _prefer_conn('M3A', ['qw2b', 'qw8b', 'ge2'])
if _m3a3_conn is not None:
    _m3a3_tabs = [r[0] for r in _m3a3_conn.execute("SELECT name FROM sqlite_master WHERE type='table'").fetchall()]
    if 'video_metadata' in _m3a3_tabs:
        _m3a3_dur = pd.read_sql_query('SELECT duration_seconds FROM video_metadata', _m3a3_conn)
        _dur_datasets['M3A'] = _m3a3_dur.duration_seconds.dropna().values
    elif _m3a_idx:
        _dur_datasets['M3A'] = np.array([e['duration_seconds'] for e in _m3a_idx
                                          if e.get('duration_seconds')], dtype=float)
    if _m3a3_conn:
        _m3a3_conn.close()

_DUR_COLORS = {'FakeVV': '#4472C4', 'GroundLie360': '#70AD47', 'M3A': '#ED7D31'}

if _dur_datasets:
    from scipy.stats import gaussian_kde
    with plt.rc_context(ma._RC):
        _fig_dur, _ax_dur = plt.subplots(figsize=(8, 4))
        _all_dur = np.concatenate(list(_dur_datasets.values()))
        _clip = np.percentile(_all_dur, 98)   # cap at 98th percentile for readability
        _xs = np.linspace(0, _clip, 400)
        for _dname, _dvals in _dur_datasets.items():
            _dclipped = _dvals[_dvals <= _clip]
            _kde_d = gaussian_kde(_dclipped)
            _ax_dur.plot(_xs, _kde_d(_xs), lw=2.0,
                         color=_DUR_COLORS.get(_dname, '#888888'),
                         label=f"{_dname}  (median={np.median(_dvals):.0f}s, N={len(_dvals)})")
            _ax_dur.fill_between(_xs, _kde_d(_xs), alpha=0.12,
                                  color=_DUR_COLORS.get(_dname, '#888888'))
        _ax_dur.set_xlabel('Duration (seconds)')
        _ax_dur.set_ylabel('Density')
        _ax_dur.set_title('Video duration distribution across datasets (capped at 98th pct.)')
        _ax_dur.legend()
        plt.tight_layout()
        _fig_dur.savefig(str(ma.FIG_DIR / 'eda_durations.png'), dpi=150, bbox_inches='tight')
        plt.show(); plt.close('all')
    print("Saved eda_durations.png")
    for _dn, _dv in _dur_datasets.items():
        print(f"  {_dn}: N={len(_dv)}  median={np.median(_dv):.1f}s  mean={np.mean(_dv):.1f}s  "
              f"p5={np.percentile(_dv,5):.1f}s  p95={np.percentile(_dv,95):.1f}s")
else:
    print("[skip] Duration overlay: no datasets with video_metadata available")


### Interpretación EDA


## 3. Geometría de los espacios de embedding


In [ ]:
# ── Plot 1: Modality gap por dataset ─────────────────────────────────────────
for ds in ['FakeVV', 'GroundLie360', 'M3A']:
    fig = ma.plot_modality_gap_umap(ds)
    plt.show()


In [ ]:
# ── Plot 2: Dataset gap (todos los embeddings, colores = dataset) ────────────
fig = ma.plot_dataset_gap_umap()
plt.show()

### 3.1 Métricas intrínsecas por encoder y modalidad (GroundLie360)


In [ ]:
# ── Geometry 3.1: anisotropy, effective_rank, intrinsic_dim per encoder×modality ──
# Datasets: FakeVV, GroundLie360, M3A
# WAVE included where DB exists and regeneration is complete

_GEOM_CONFIG = {
    'FakeVV':       {'models': ['ge2', 'qw2b', 'qw8b'],         'modalities': ['video', 'text_title', 'transcript']},
    'GroundLie360': {'models': ['ge2', 'qw2b', 'qw8b', 'wave'], 'modalities': ['video', 'text_title', 'transcript']},
    'M3A':          {'models': ['ge2', 'qw2b', 'qw8b', 'wave'], 'modalities': ['video', 'text_summary', 'transcript']},
}

_geom_rows = []

for _ds, _cfg in _GEOM_CONFIG.items():
    for _gm in _cfg['models']:
        _info_gm = ma.available(_ds, _gm)
        if not _info_gm.get('exists'):
            print(f'[skip] {_ds}/{_gm}: DB not found')
            continue
        _mods_gm = _info_gm.get('modalities', {})
        try:
            _conn_gm = ma.connect(ma.db_path(_ds, _gm))
        except Exception as ex:
            print(f'[skip] {_ds}/{_gm}: connect error: {ex}')
            continue

        for _mod in _cfg['modalities']:
            if _mod not in _mods_gm:
                print(f'[skip] {_ds}/{_gm}/{_mod}: modality not present')
                continue
            try:
                _emb = ma.load_globals(_conn_gm, _mod)
                if not _emb:
                    print(f'[skip] {_ds}/{_gm}/{_mod}: empty')
                    continue
                _aniso   = ma.anisotropy(_emb)
                _effrank = ma.effective_rank(_emb)
                _idim    = ma.intrinsic_dim_twonn(_emb, sample=2000, seed=0)
                _geom_rows.append({
                    'dataset':        _ds,
                    'encoder':        _gm,
                    'modality':       _mod,
                    'N':              len(_emb),
                    'anisotropy':     round(_aniso,   4),
                    'effective_rank': round(_effrank, 1),
                    'intrinsic_dim':  round(_idim,    1),
                })
                print(f'  {_ds:12s}  {ma.MODEL_DISPLAY[_gm]:15s}  {_mod:15s}  '
                      f'aniso={_aniso:.4f}  eff_rank={_effrank:.1f}  idim={_idim:.1f}')
            except Exception as ex:
                print(f'[skip] {_ds}/{_gm}/{_mod}: error: {ex}')

        _conn_gm.close()

df_geom = pd.DataFrame(_geom_rows)
if not df_geom.empty:
    print('\n=== Per-encoder intrinsic metrics (all datasets) ===')
    display(df_geom.set_index(['dataset', 'encoder', 'modality']))
else:
    print('[skip] Geom 3.1: no results')


In [ ]:
# ── Geometry 3.1 figures: grouped bars of anisotropy and effective_rank (GroundLie360) ──
_geo_gl = df_geom[df_geom.dataset == 'GroundLie360'] if ('dataset' in df_geom.columns) else df_geom
_GEOM_MODALITIES = list(_geo_gl.modality.unique())
if not _geo_gl.empty:
    # anisotropy
    _aniso_data = {}
    for _mod in _GEOM_MODALITIES:
        _sub = _geo_gl[_geo_gl.modality == _mod]
        if not _sub.empty:
            _aniso_data[_mod] = {r['encoder']: r['anisotropy'] for _, r in _sub.iterrows()}
    if _aniso_data:
        _fig_aniso = ma.grouped_bar(
            _aniso_data,
            ylabel='Anisotropy (mean random-pair cosine)',
            title='Geometry — Anisotropy by encoder × modality (GroundLie360)',
            fname='geom_anisotropy.png',
            baseline=0.0,
        )
        plt.show(); plt.close('all')

    # effective_rank
    _effr_data = {}
    for _mod in _GEOM_MODALITIES:
        _sub = _geo_gl[_geo_gl.modality == _mod]
        if not _sub.empty:
            _effr_data[_mod] = {r['encoder']: r['effective_rank'] for _, r in _sub.iterrows()}
    if _effr_data:
        _fig_effr = ma.grouped_bar(
            _effr_data,
            ylabel='Effective rank (exp H(singular values))',
            title='Geometry — Effective rank by encoder × modality (GroundLie360)',
            fname='geom_effrank.png',
            baseline=None,
        )
        plt.show(); plt.close('all')
    print("Saved geom_anisotropy.png, geom_effrank.png")
else:
    print("[skip] Geometry 3.1 figures: no GroundLie360 data")

In [ ]:
# ── Geometry 3.1 VIZ: anisotropy por dataset — solo necesita df_geom ──────────
for _ds in ['FakeVV', 'GroundLie360', 'M3A']:
    _sub_ds = df_geom[df_geom.dataset == _ds] if 'dataset' in df_geom.columns else df_geom
    # ── Geometry 3.1 VIZ: anisotropy por dataset — solo necesita df_geom ──────────
    for _ds in ['FakeVV', 'GroundLie360', 'M3A']:
        _sub_ds = df_geom[df_geom.dataset == _ds] if 'dataset' in df_geom.columns else df_geom
        if _sub_ds.empty:
            print(f'[skip] {_ds}: no data in df_geom')
            continue
        _data = {}
        for _mod in _sub_ds.modality.unique():
            _rows = _sub_ds[_sub_ds.modality == _mod]
            if not _rows.empty:
                _data[_mod] = {r['encoder']: r['anisotropy'] for _, r in _rows.iterrows()}
        if _data:
            ma.grouped_bar(
                _data,
                ylabel='Anisotropy (mean random-pair cosine)',
                title=f'Anisotropy — {_ds}',
                fname=f'geom_anisotropy_{_ds.lower()}.png',
                baseline=0.0,
            )
            plt.show(); plt.close('all')
        print(f'[skip] {_ds}: no data in df_geom')
        continue
    _data = {}
    for _mod in _sub_ds.modality.unique():
        _rows = _sub_ds[_sub_ds.modality == _mod]
        if not _rows.empty:
            _data[_mod] = {r['encoder']: r['anisotropy'] for _, r in _rows.iterrows()}
    if _data:
        ma.grouped_bar(
            _data,
            ylabel='Anisotropy (mean random-pair cosine)',
            title=f'Anisotropy — {_ds}',
            fname=f'geom_anisotropy_{_ds.lower()}.png',
            baseline=0.0,
        )
        plt.show(); plt.close('all')

In [ ]:
# ── Geometry 3.1 VIZ: effective_rank por dataset — solo necesita df_geom ──────
for _ds in ['FakeVV', 'GroundLie360', 'M3A']:
    _sub_ds = df_geom[df_geom.dataset == _ds] if 'dataset' in df_geom.columns else df_geom
    if _sub_ds.empty:
        print(f'[skip] {_ds}: no data in df_geom')
        continue
    _data = {}
    for _mod in _sub_ds.modality.unique():
        _rows = _sub_ds[_sub_ds.modality == _mod]
        if not _rows.empty:
            _data[_mod] = {r['encoder']: r['effective_rank'] for _, r in _rows.iterrows()}
    if _data:
        ma.grouped_bar(
            _data,
            ylabel='Effective rank (exp H(singular values))',
            title=f'Effective rank — {_ds}',
            fname=f'geom_effrank_{_ds.lower()}.png',
        )
        plt.show(); plt.close('all')


### 3.2 Alignment & Uniformity (pares cross-modal en GroundLie360)


In [ ]:
# ── Geometry 3.2: Alignment & Uniformity ─────────────────────────────────────
# Positive pairs: video↔text_title, video↔transcript
_AU_PAIRS = [('video', 'text_title'), ('video', 'transcript')]
_GEOM_MODELS = ['ge2', 'qw2b', 'qw8b', 'wave']   # GroundLie360 (el guard salta modelos sin vídeo)

_au_rows = []

for _gm in _GEOM_MODELS:
    _info_gm = ma.available('GroundLie360', _gm)
    _mods_gm = _info_gm.get('modalities', {}) if _info_gm.get('exists') else {}
    if not _info_gm.get('exists') or 'video' not in _mods_gm:
        print(f"[skip] Geom AU {_gm}: DB not found or video missing")
        continue
    try:
        _conn_gm = ma.connect(ma.db_path('GroundLie360', _gm))
        _Vi_au = ma.load_globals(_conn_gm, 'video')
    except Exception as ex:
        print(f"[skip] Geom AU {_gm}: load error: {ex}")
        continue

    for _ma_mod, _mb_mod in _AU_PAIRS:
        if _mb_mod not in _mods_gm:
            print(f"[skip] Geom AU {_gm} {_ma_mod}↔{_mb_mod}: {_mb_mod} missing")
            continue
        try:
            _Mb_au = ma.load_globals(_conn_gm, _mb_mod)
            _au = ma.alignment_uniformity(_Vi_au, _Mb_au, sample=2000, seed=0)
            _au_rows.append({
                'encoder':     _gm,
                'pair':        f'{_ma_mod}↔{_mb_mod}',
                'alignment':   round(_au['alignment'],   4),
                'uniformity':  round(_au['uniformity'],  4),
            })
            print(f"  {ma.MODEL_DISPLAY[_gm]:15s}  {_ma_mod}↔{_mb_mod:12s}  "
                  f"align={_au['alignment']:.4f}  uniform={_au['uniformity']:.4f}")
        except Exception as ex:
            print(f"[skip] Geom AU {_gm} {_ma_mod}↔{_mb_mod}: {ex}")

    _conn_gm.close()

df_au = pd.DataFrame(_au_rows)
if not df_au.empty:
    display(df_au.set_index(['encoder', 'pair']))

    # Scatter: alignment(x) vs uniformity(y), colored by encoder
    with plt.rc_context(ma._RC):
        _fig_au, _ax_au = plt.subplots(figsize=(6, 5))
        _pair_markers = {'video↔text_title': 'o', 'video↔transcript': 's'}
        for _, _row_au in df_au.iterrows():
            _ax_au.scatter(
                _row_au['alignment'], _row_au['uniformity'],
                color=ma.MODEL_COLOR.get(_row_au['encoder'], '#888'),
                marker=_pair_markers.get(_row_au['pair'], 'D'),
                s=90, edgecolors='white', linewidths=0.5,
                label=f"{ma.MODEL_DISPLAY[_row_au['encoder']]} · {_row_au['pair']}",
                zorder=3,
            )
        _ax_au.set_xlabel('Alignment (lower = better aligned pairs)')
        _ax_au.set_ylabel('Uniformity (log; lower = more uniform on sphere)')
        _ax_au.set_title('Alignment & Uniformity per encoder × pair (GroundLie360)')
        _ax_au.legend(fontsize=7, ncol=1, framealpha=0.85)
        plt.tight_layout()
        _fig_au.savefig(str(ma.FIG_DIR / 'geom_align_uniform.png'), dpi=150, bbox_inches='tight')
        plt.show(); plt.close('all')
    print("Saved geom_align_uniform.png")
else:
    print("[skip] Alignment & Uniformity: no results")


In [ ]:
# ── Geometry 3.2: Alignment & Uniformity — FakeVV pares reales ───────────────

def _alignment(vecs_a, vecs_b):
    """E[||u-v||²] con u,v L2-normalizados."""
    a = np.array([v / (np.linalg.norm(v) + 1e-9) for v in vecs_a], dtype=np.float32)
    b = np.array([v / (np.linalg.norm(v) + 1e-9) for v in vecs_b], dtype=np.float32)
    return float(np.mean(np.sum((a - b)**2, axis=1)))

def _uniformity(vecs, n_pairs=5000, seed=42):
    """log E[exp(-2||u-v||²)] sobre pares aleatorios."""
    rng = np.random.default_rng(seed)
    mat = np.array([v / (np.linalg.norm(v) + 1e-9) for v in vecs], dtype=np.float32)
    N = len(mat)
    idx_a = rng.integers(0, N, n_pairs)
    idx_b = rng.integers(0, N, n_pairs)
    sq_dist = np.sum((mat[idx_a] - mat[idx_b])**2, axis=1)
    return float(np.log(np.mean(np.exp(-2 * sq_dist))))

_au_rows = []
_AU_MODELS = ['ge2', 'qw2b', 'qw8b']

for _gm in _AU_MODELS:
    _info = ma.available('FakeVV', _gm)
    if not _info.get('exists'):
        print(f'[skip] FakeVV/{_gm}: DB not found')
        continue
    _mods = _info.get('modalities', {})
    if 'video' not in _mods or 'text_real' not in _mods:
        print(f'[skip] FakeVV/{_gm}: video o text_real ausente')
        continue

    _conn = ma.connect(ma.db_path('FakeVV', _gm))
    _vid  = ma.load_globals(_conn, 'video')
    _titl = ma.load_globals(_conn, 'text_real')
    _conn.close()

    _common = sorted(set(_vid) & set(_titl))
    _vecs_vid  = [_vid[rid]  for rid in _common]
    _vecs_titl = [_titl[rid] for rid in _common]

    _aln = _alignment(_vecs_vid, _vecs_titl)
    _uni = _uniformity(_vecs_vid)

    _au_rows.append({
        'encoder':    _gm,
        'N_pairs':    len(_common),
        'alignment':  round(_aln, 4),
        'uniformity': round(_uni, 4),
    })
    print(f'  {ma.MODEL_DISPLAY[_gm]:15s}  N={len(_common)}  '
          f'alignment={_aln:.4f}  uniformity={_uni:.4f}')

if not _au_rows:
    print('[ERROR] No se procesó ningún modelo.')
else:
    df_au = pd.DataFrame(_au_rows).set_index('encoder')
    df_au.index = [ma.MODEL_DISPLAY[m] for m in df_au.index]
    display(df_au)


In [ ]:
# ── Geometry 3.2 VIZ: Alignment & Uniformity — FakeVV pares reales ────────────
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(10, 4))

_colors = [ma.MODEL_COLOR[m] for m in _AU_MODELS if ma.MODEL_DISPLAY[m] in df_au.index]
# Alignment (menor = mejor)
ax = axes[0]
_vals = df_au['alignment'].values
bars = ax.bar(df_au.index, _vals, color=_colors, edgecolor='white', linewidth=0.6)
for bar, v in zip(bars, _vals):
    ax.text(bar.get_x() + bar.get_width()/2, v + 0.005, f'{v:.4f}',
            ha='center', va='bottom', fontsize=8)
ax.set_ylabel('Alignment  E[||u-v||²]')
ax.set_title('Alignment (↓ mejor)\nvideo ↔ text_real, FakeVV')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

# Uniformity (más negativo = mejor)
ax = axes[1]
_vals = df_au['uniformity'].values
bars = ax.bar(df_au.index, _vals, color=_colors, edgecolor='white', linewidth=0.6)
for bar, v in zip(bars, _vals):
    ax.text(bar.get_x() + bar.get_width()/2, v - 0.02, f'{v:.4f}',
            ha='center', va='top', fontsize=8)
ax.set_ylabel('Uniformity  log E[exp(-2||u-v||²)]')
ax.set_title('Uniformity (↓ mejor)\nembeddings de vídeo, FakeVV')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.suptitle('Alignment & Uniformity — FakeVV (pares reales)', fontsize=12, y=1.02)
plt.tight_layout()
plt.savefig(str(ma.FIG_DIR / 'geom_alignment_uniformity_fakevv.png'), dpi=150, bbox_inches='tight')
plt.show()

### 3.3 CKA cross-encoder (M3A, N=8000)


In [ ]:
# ── Geometry 3.3: Linear CKA cross-encoder on M3A (N=8000) ───────────────────
# Models: ge2, qw2b, qw8b — WAVE excluded (regeneration)
# Modalities: video, text_summary
# Subsample 8000 common ids across all 3 models (seed=0)

_CKA_MODELS = ['ge2', 'qw2b', 'qw8b']
_CKA_N      = 8000

# Load embeddings for each model
_cka_embs = {}   # model -> modality -> {raw_id: vec}
for _cm in _CKA_MODELS:
    _info_cm = ma.available('M3A', _cm)
    _mods_cm = _info_cm.get('modalities', {}) if _info_cm.get('exists') else {}
    if not _info_cm.get('exists') or 'video' not in _mods_cm:
        print(f"[skip] CKA {_cm}: M3A DB not found or video missing")
        continue
    try:
        _conn_cm = ma.connect(ma.db_path('M3A', _cm))
        _cka_embs[_cm] = {}
        _cka_embs[_cm]['video'] = ma.load_globals(_conn_cm, 'video')
        if 'text_summary' in _mods_cm:
            _cka_embs[_cm]['text_summary'] = ma.load_globals(_conn_cm, 'text_summary')
        _conn_cm.close()
        print(f"  Loaded M3A {_cm}: N_video={len(_cka_embs[_cm]['video'])}")
    except Exception as ex:
        print(f"[skip] CKA {_cm}: load error: {ex}")

# Find common ids across all available models (video modality)
_cka_common_video = set.intersection(*[set(_cka_embs[m]['video'].keys())
                                        for m in _cka_embs if 'video' in _cka_embs[m]]) \
                    if _cka_embs else set()
print(f"Common video ids across {list(_cka_embs.keys())}: {len(_cka_common_video)}")

_rng_cka = np.random.default_rng(0)
_models_with_video = [m for m in _CKA_MODELS if m in _cka_embs and 'video' in _cka_embs[m]]
_cka_video_mat = np.full((len(_models_with_video), len(_models_with_video)), np.nan)
_cka_n_used = {}   # (mi, mj) -> N actually used

for _i, _mi in enumerate(_models_with_video):
    _cka_video_mat[_i, _i] = 1.0
    for _j, _mj in enumerate(_models_with_video):
        if _j <= _i:
            continue
        # Common ids between this specific pair
        _pair_common = sorted(set(_cka_embs[_mi]['video'].keys()) & set(_cka_embs[_mj]['video'].keys()))
        _n_pair = len(_pair_common)
        _n_use  = min(_n_pair, _CKA_N)
        if _n_use < 100:
            print(f"[skip] CKA {_mi}↔{_mj}: only {_n_pair} common ids")
            continue
        _ids_ij = sorted(_rng_cka.choice(_pair_common, size=_n_use, replace=False)) if _n_pair > _n_use else _pair_common
        _cka_n_used[(_mi, _mj)] = len(_ids_ij)
        try:
            _Xi = np.stack([_cka_embs[_mi]['video'][rid].astype(np.float32) for rid in _ids_ij])
            _Xj = np.stack([_cka_embs[_mj]['video'][rid].astype(np.float32) for rid in _ids_ij])
            _v = ma.linear_cka(_Xi, _Xj)
            _cka_video_mat[_i, _j] = _v
            _cka_video_mat[_j, _i] = _v
            print(f"  CKA {_mi}↔{_mj}: N={len(_ids_ij)}  CKA={_v:.4f}")
        except Exception as ex:
            print(f"[skip] CKA {_mi}↔{_mj}: {ex}")

if not np.all(np.isnan(_cka_video_mat) | (_cka_video_mat == 1.0)):
    _cka_df = pd.DataFrame(_cka_video_mat,
                            index=[ma.MODEL_DISPLAY[m] for m in _models_with_video],
                            columns=[ma.MODEL_DISPLAY[m] for m in _models_with_video])
    _n_note = {f"{ma.MODEL_DISPLAY[a]}↔{ma.MODEL_DISPLAY[b]}": n for (a,b),n in _cka_n_used.items()}
    print("\nN used per pair:", _n_note)
    print("\n=== Cross-encoder video CKA on M3A ===")
    display(_cka_df.round(3))

    # Heatmap
    with plt.rc_context(ma._RC):
        _fig_cka, _ax_cka = plt.subplots(figsize=(5, 4))
        _im_cka = _ax_cka.imshow(_cka_video_mat, cmap='Blues', vmin=0, vmax=1, aspect='auto')
        _labs_cka = [ma.MODEL_DISPLAY[m] for m in _models_with_video]
        _ax_cka.set_xticks(range(len(_labs_cka)))
        _ax_cka.set_yticks(range(len(_labs_cka)))
        _ax_cka.set_xticklabels(_labs_cka, rotation=25, ha='right', fontsize=9)
        _ax_cka.set_yticklabels(_labs_cka, fontsize=9)
        for _i2 in range(len(_models_with_video)):
            for _j2 in range(len(_models_with_video)):
                _v_cka = _cka_video_mat[_i2, _j2]
                if not np.isnan(_v_cka):
                    _ax_cka.text(_j2, _i2, f'{_v_cka:.3f}', ha='center', va='center',
                                 fontsize=10, color='white' if _v_cka > 0.7 else 'black')
        _n_pairs_str = ', '.join(f'{k}={v}' for k,v in _n_note.items())
        plt.colorbar(_im_cka, ax=_ax_cka, shrink=0.8, label='Linear CKA')
        _ax_cka.set_title(f'Cross-encoder CKA — video modality (M3A)', pad=10)
        plt.tight_layout()
        _fig_cka.savefig(str(ma.FIG_DIR / 'geom_cka_video.png'), dpi=150, bbox_inches='tight')
        plt.show(); plt.close('all')
    print("Saved geom_cka_video.png")
else:
    print("[skip] CKA heatmap: no valid off-diagonal entries")
    _cka_df = pd.DataFrame()

# Optional: intra-encoder modality CKA for qw2b (video vs text_summary vs transcript)
if 'qw2b' in _cka_embs and 'text_summary' in _cka_embs.get('qw2b', {}):
    _qw2b_v  = _cka_embs['qw2b']['video']
    _qw2b_t  = _cka_embs['qw2b']['text_summary']
    _common_vt = sorted(set(_qw2b_v.keys()) & set(_qw2b_t.keys()))
    _n_vt = min(len(_common_vt), _CKA_N)
    if _n_vt > 100:
        _ids_vt = sorted(_rng_cka.choice(_common_vt, size=_n_vt, replace=False)) if len(_common_vt) > _n_vt else _common_vt
        try:
            _Xv2 = np.stack([_qw2b_v[rid].astype(np.float32) for rid in _ids_vt])
            _Xt2 = np.stack([_qw2b_t[rid].astype(np.float32) for rid in _ids_vt])
            _cka_vt2 = ma.linear_cka(_Xv2, _Xt2)
            print(f"Qwen3-VL 2B  video↔text_summary CKA (N={len(_ids_vt)}): {_cka_vt2:.4f}")
        except Exception as ex:
            print(f"[skip] Intra-encoder CKA qw2b video↔text_summary: {ex}")


### 3.4 Correlación geometría ↔ downstream


In [ ]:
# ── Geometry 3.4: Gap + Anisotropy + R@1 por modelo ─────────────────────────
# Objetivo: justificar la comparacion de los modelos mostrando espacios distintos
# Gap + anisotropia: GroundLie360/video (todos los modelos disponibles)
# R@1: FakeVV text_real (pares reales limpios, sin ruido de etiquetas)

_GEOM_MODELS = ['ge2', 'qw2b', 'qw8b']
_downstream_rows = []

for _dm in _GEOM_MODELS:
    # Geometria desde df_geom (GroundLie360, video)
    _geom_row = (
        df_geom[(df_geom.dataset == 'GroundLie360') &
                (df_geom.encoder == _dm) &
                (df_geom.modality == 'video')]
        if 'dataset' in df_geom.columns else
        df_geom[(df_geom.encoder == _dm) & (df_geom.modality == 'video')]
    )
    _aniso_v = float(_geom_row.iloc[0]['anisotropy'])    if not _geom_row.empty else float('nan')
    _effr_v  = float(_geom_row.iloc[0]['effective_rank']) if not _geom_row.empty else float('nan')

    # Modality gap video<->text_title en GroundLie360
    _gap_vt = float('nan')
    _info_gl = ma.available('GroundLie360', _dm)
    if _info_gl.get('exists'):
        _mods_gl = _info_gl.get('modalities', {})
        if 'video' in _mods_gl and 'text_title' in _mods_gl:
            _conn_gl = ma.connect(ma.db_path('GroundLie360', _dm))
            _Vi = ma.load_globals(_conn_gl, 'video')
            _Ti = ma.load_globals(_conn_gl, 'text_title')
            _conn_gl.close()
            _gap_res = ma.modality_gap(_Vi, _Ti, intra_n=500, seed=0)
            _gap_vt  = _gap_res['gap_euclidean']

    # R@1: retrieval video->text_real en FakeVV (pares reales)
    _r1 = float('nan')
    _info_fv = ma.available('FakeVV', _dm)
    if _info_fv.get('exists'):
        _mods_fv = _info_fv.get('modalities', {})
        if 'video' in _mods_fv and 'text_real' in _mods_fv:
            _conn_fv = ma.connect(ma.db_path('FakeVV', _dm))
            _Vf = ma.load_globals(_conn_fv, 'video')
            _Tf = ma.load_globals(_conn_fv, 'text_real')
            _conn_fv.close()
            _retr = ma.retrieval_metrics(_Vf, _Tf, sample_n=len(set(_Vf) & set(_Tf)), seed=0)
            _r1   = _retr['R@1']

    _downstream_rows.append({
        'encoder':        _dm,
        'anisotropy':     round(_aniso_v, 4),
        'effective_rank': round(_effr_v,  1),
        'gap_V_T':        round(_gap_vt,  4),
        'R@1_FakeVV':     round(_r1,      3),
    })
    print(f'  {ma.MODEL_DISPLAY[_dm]:15s}  '
          f'aniso={_aniso_v:.4f}  eff_rank={_effr_v:.1f}  '
          f'gap={_gap_vt:.4f}  R@1_FakeVV={_r1:.3f}')

df_downstream = pd.DataFrame(_downstream_rows).set_index('encoder')
df_downstream.index = [ma.MODEL_DISPLAY[m] for m in df_downstream.index]
print('\n=== Geometria vs Alineamiento por encoder ===')
display(df_downstream)


### Interpretación de la geometría


## 4. Dataset A — FakeVV (línea de base controlada)


In [ ]:
# ── FakeVV: cargar índice de categorías ───────────────────────────────────────
FAKEVV_INDEX = REPO / "experiments/FakeVV_testset/google_embeddings2/test_index.json"
if FAKEVV_INDEX.exists():
    cat_by_id = {e['raw_id']: e.get('category', 'unknown')
                 for e in json.load(open(FAKEVV_INDEX, encoding='utf-8'))}
    print(f"FakeVV index: {len(cat_by_id)} entries")
    from collections import Counter
    print("Categorías:", dict(Counter(cat_by_id.values())))
else:
    print("[skip] FakeVV index not found")
    cat_by_id = {}


### E1 — cos(video, title_real) vs cos(video, title_fake)


In [ ]:
e1_fakevv = []
e1_by_cat = {}  # model -> category -> [acc per id]

for model in ma.MODELS_BY_DATASET['FakeVV']:
    info = ma.available('FakeVV', model)
    mods = info.get('modalities', {}) if info.get('exists') else {}
    has_video = 'video' in mods
    has_real  = 'text_real' in mods
    has_fake  = 'text_fake' in mods
    if not (info.get('exists') and has_video and has_real and has_fake):
        print(f"[skip] FakeVV E1 {model}: modalities video/text_real/text_fake not all present "
              f"(exists={info.get('exists')}, mods={list(mods.keys())})")
        continue
    conn = ma.connect(ma.db_path('FakeVV', model))
    vid   = ma.load_globals(conn, 'video')
    t_rl  = ma.load_globals(conn, 'text_real')
    t_fk  = ma.load_globals(conn, 'text_fake')
    conn.close()

    ids = sorted(set(vid) & set(t_rl) & set(t_fk))
    real_ = [ma.cos(vid[r], t_rl[r]) for r in ids]
    fake_ = [ma.cos(vid[r], t_fk[r]) for r in ids]
    row = ma.summarize_paired(real_, fake_, f'FakeVV-E1-{model}')
    row['model'] = model
    e1_fakevv.append(row)

    # per-category
    e1_by_cat[model] = {}
    for rid in ids:
        cat = cat_by_id.get(rid, 'unknown')
        e1_by_cat[model].setdefault(cat, {'real': [], 'fake': []})
        e1_by_cat[model][cat]['real'].append(ma.cos(vid[rid], t_rl[rid]))
        e1_by_cat[model][cat]['fake'].append(ma.cos(vid[rid], t_fk[rid]))

    print(f"[E1] {ma.MODEL_DISPLAY[model]:15s}  N={row['n']:4d}  "
          f"acc={row['acc']:.3f}  diff={row['mean_diff']:+.4f}  p={row['p']}")

df_e1_fakevv = pd.DataFrame(e1_fakevv) if e1_fakevv else pd.DataFrame()
if not df_e1_fakevv.empty:
    display(df_e1_fakevv.set_index('model'))


### E2 — cos(transcript, title_real) vs cos(transcript, title_fake)


In [ ]:
e2_fakevv = []

for model in ma.MODELS_BY_DATASET['FakeVV']:
    info = ma.available('FakeVV', model)
    mods = info.get('modalities', {}) if info.get('exists') else {}
    if not (info.get('exists') and 'transcript' in mods and 'text_real' in mods and 'text_fake' in mods):
        print(f"[skip] FakeVV E2 {model}: transcript or text_real/fake missing")
        continue
    conn = ma.connect(ma.db_path('FakeVV', model))
    tr    = ma.load_globals(conn, 'transcript')
    t_rl  = ma.load_globals(conn, 'text_real')
    t_fk  = ma.load_globals(conn, 'text_fake')
    conn.close()

    ids = sorted(set(tr) & set(t_rl) & set(t_fk))
    real_ = [ma.cos(tr[r], t_rl[r]) for r in ids]
    fake_ = [ma.cos(tr[r], t_fk[r]) for r in ids]
    row = ma.summarize_paired(real_, fake_, f'FakeVV-E2-{model}')
    row['model'] = model
    e2_fakevv.append(row)
    print(f"[E2] {ma.MODEL_DISPLAY[model]:15s}  N={row['n']:4d}  "
          f"acc={row['acc']:.3f}  diff={row['mean_diff']:+.4f}  p={row['p']}")

df_e2_fakevv = pd.DataFrame(e2_fakevv) if e2_fakevv else pd.DataFrame()
if not df_e2_fakevv.empty:
    display(df_e2_fakevv.set_index('model'))


In [ ]:
# ── Figures: FakeVV video/transcript ↔ title discrimination (2AFC) ────────────
e1_acc = {r['model']: r['acc'] for r in e1_fakevv}
e2_acc = {r['model']: r['acc'] for r in e2_fakevv}

if e1_acc:
    fig = ma.grouped_bar({'Video ↔ Title': e1_acc, 'Transcript ↔ Title': e2_acc},
                         ylabel='2AFC accuracy (real vs. fake)',
                         title='FakeVV — Real-vs-Fake Title Discrimination (2AFC, chance = 0.5)',
                         fname='fakevv_e1e2_accuracy.png', baseline=0.5, pct=True)
    plt.show(); plt.close('all')

# Per-category 2AFC accuracy (video ↔ title)
if e1_by_cat:
    cats = sorted({c for m in e1_by_cat for c in e1_by_cat[m]})
    cat_acc = {cat: {} for cat in cats}
    for model, cdict in e1_by_cat.items():
        for cat, arrs in cdict.items():
            rl, fk = np.array(arrs['real']), np.array(arrs['fake'])
            cat_acc[cat][model] = float(np.mean(rl > fk))
    fig = ma.grouped_bar(cat_acc,
                         ylabel='2AFC accuracy (video ↔ title)',
                         title='FakeVV — Video↔Title Discrimination by Entity-Swap Category (2AFC, chance = 0.5)',
                         fname='fakevv_e1_by_category.png', baseline=0.5, pct=True)
    plt.show(); plt.close('all')

### E1b / E2b — Clasificación real con umbral (train/test)


In [ ]:
from collections import defaultdict

# ── E1b/E2b: clasificación real con umbral aprendido en train ────────────────
N_REPS, TRAIN_FRAC = 50, 0.7

def fit_threshold(scores, labels):
    '''Umbral que maximiza accuracy en train para la regla `fake si cos < thr`.'''
    s = np.asarray(scores, float); l = np.asarray(labels, int)
    order = np.argsort(s, kind='mergesort')
    s_sorted, l_sorted = s[order], l[order]
    n, n_fake = len(s), int(l.sum())
    fake_in_k = np.concatenate([[0], np.cumsum(l_sorted)])   # fakes among k smallest
    k = np.arange(n + 1)
    correct = fake_in_k + ((n - n_fake) - (k - fake_in_k))   # fakes <thr + reals >=thr
    best_k = int(np.argmax(correct))
    if best_k == 0:   return float(s_sorted[0] - 1e-9)
    if best_k == n:   return float(s_sorted[-1] + 1e-9)
    return float((s_sorted[best_k - 1] + s_sorted[best_k]) / 2)

e1b_rows, e1b_cat = [], {}   # e1b_cat: (exp, model) -> {cat: mean acc}

for exp_tag, anchor_mod in [("E1b video↔title", "video"), ("E2b transcript↔title", "transcript")]:
    for model in ma.MODELS_BY_DATASET['FakeVV']:
        info = ma.available('FakeVV', model)
        mods = info.get('modalities', {}) if info.get('exists') else {}
        if not (info.get('exists') and anchor_mod in mods and 'text_real' in mods and 'text_fake' in mods):
            print(f"[skip] FakeVV {exp_tag} {model}")
            continue
        conn = ma.connect(ma.db_path('FakeVV', model))
        anchor = ma.load_globals(conn, anchor_mod)
        t_rl   = ma.load_globals(conn, 'text_real')
        t_fk   = ma.load_globals(conn, 'text_fake')
        conn.close()

        ids   = sorted(set(anchor) & set(t_rl) & set(t_fk))
        cos_r = {r: ma.cos(anchor[r], t_rl[r]) for r in ids}
        cos_f = {r: ma.cos(anchor[r], t_fk[r]) for r in ids}
        by_cat = defaultdict(list)
        for r in ids:
            by_cat[cat_by_id.get(r, 'unknown')].append(r)

        accs, thrs = [], []
        cat_accs = defaultdict(list)
        for rep in range(N_REPS):
            rng = np.random.default_rng(1000 + rep)
            train_ids, test_ids = [], []
            for cat, rs in by_cat.items():          # split por vídeo, estratificado por categoría
                rs = list(rs); rng.shuffle(rs)
                k = int(round(TRAIN_FRAC * len(rs)))
                train_ids += rs[:k]; test_ids += rs[k:]

            thr = fit_threshold(
                [cos_r[r] for r in train_ids] + [cos_f[r] for r in train_ids],
                [0] * len(train_ids) + [1] * len(train_ids))

            te_s = np.array([cos_r[r] for r in test_ids] + [cos_f[r] for r in test_ids])
            te_l = np.array([0] * len(test_ids) + [1] * len(test_ids))
            pred = (te_s < thr).astype(int)
            accs.append(float((pred == te_l).mean()))
            thrs.append(thr)

            cat_arr = np.array([cat_by_id.get(r, 'unknown') for r in test_ids] * 2)
            for cat in by_cat:
                m = cat_arr == cat
                if m.any():
                    cat_accs[cat].append(float((pred[m] == te_l[m]).mean()))

        row = {"exp": exp_tag, "model": model,
               "test_acc_mean": round(float(np.mean(accs)), 4),
               "test_acc_std":  round(float(np.std(accs)), 4),
               "thr_mean":      round(float(np.mean(thrs)), 4),
               "n_videos": len(ids), "n_test_ex": 2 * (len(ids) - int(round(TRAIN_FRAC * len(ids))))}
        e1b_rows.append(row)
        e1b_cat[(exp_tag, model)] = {c: float(np.mean(a)) for c, a in sorted(cat_accs.items())}
        print(f"[{exp_tag}] {ma.MODEL_DISPLAY[model]:13s} test acc={row['test_acc_mean']:.3f}±{row['test_acc_std']:.3f}  "
              f"thr={row['thr_mean']:+.3f}  N_vid={row['n_videos']}")

df_e1b = pd.DataFrame(e1b_rows)
print()
print(df_e1b.to_string(index=False))

print("\nAccuracy en test por categoría (E1b):")
cats_all = sorted({c for (e, m), d in e1b_cat.items() if e.startswith('E1b') for c in d})
hdr = f"{'model':<8}" + "".join(f"{c:>14}" for c in cats_all)
print(hdr)
for model in ma.MODELS_BY_DATASET['FakeVV']:
    d = e1b_cat.get(("E1b video↔title", model), {})
    if d:
        print(f"{model:<8}" + "".join(f"{d.get(c, float('nan')):>14.3f}" for c in cats_all))
print("\nAccuracy en test por categoría (E2b):")
for model in ma.MODELS_BY_DATASET['FakeVV']:
    d = e1b_cat.get(("E2b transcript↔title", model), {})
    if d:
        print(f"{model:<8}" + "".join(f"{d.get(c, float('nan')):>14.3f}" for c in cats_all))

# ── Figuras: paired (ranking) vs threshold, y por categoría ──────────────────
e1b_acc = {r['model']: r['test_acc_mean'] for r in e1b_rows if r['exp'].startswith('E1b')}
e2b_acc = {r['model']: r['test_acc_mean'] for r in e1b_rows if r['exp'].startswith('E2b')}
groups = {}
if 'e1_acc' in globals() and e1_acc:
    groups['E1 paired (ranking)'] = e1_acc
if e1b_acc:
    groups['E1b threshold (test)'] = e1b_acc
if 'e2_acc' in globals() and e2_acc:
    groups['E2 paired (ranking)'] = e2_acc
if e2b_acc:
    groups['E2b threshold (test)'] = e2b_acc
if groups:
    fig = ma.grouped_bar(groups, ylabel='Accuracy',
                         title='FakeVV — Paired ranking vs threshold classification',
                         fname='fakevv_paired_vs_threshold.png', baseline=0.5, pct=True)
    plt.show(); plt.close('all')

cat_groups = {}
for cat in cats_all:
    d = {m: e1b_cat[('E1b video↔title', m)][cat]
         for m in ma.MODELS_BY_DATASET['FakeVV']
         if ('E1b video↔title', m) in e1b_cat and cat in e1b_cat[('E1b video↔title', m)]}
    if d:
        cat_groups[cat] = d
if cat_groups:
    fig = ma.grouped_bar(cat_groups, ylabel='E1b test accuracy',
                         title='FakeVV E1b — Threshold accuracy by category',
                         fname='fakevv_e1b_by_category.png', baseline=0.5, pct=True)
    plt.show(); plt.close('all')

In [ ]:
FV_MODELS = ['qw2b', 'qw8b', 'wave', 'ge2']

def _l2(v):
    n = np.linalg.norm(v); return v / n if n > 1e-12 else v
def _area3(a, b, c):                       # área real del triángulo (TRIANGLE)
    a, b, c = _l2(a), _l2(b), _l2(c); u, v = b - a, c - a
    return 0.5 * np.sqrt(max(float(u@u*(v@v) - (u@v)**2), 0.0))

fv_2afc = []
for model in FV_MODELS:
    mods = ma.available('FakeVV', model).get('modalities', {})
    if not {'text_real', 'text_fake', 'video', 'transcript'} <= set(mods):
        print(f"[skip] FakeVV 2AFC {model}: sin transcript (WAVE guarda audio/AV)"); continue
    conn = ma.connect(ma.db_path('FakeVV', model))
    TR, TF = ma.load_globals(conn, 'text_real'), ma.load_globals(conn, 'text_fake')
    Vi, Tx = ma.load_globals(conn, 'video'), ma.load_globals(conn, 'transcript')
    conn.close()
    ids = set(TR) & set(TF) & set(Vi) & set(Tx)
    e1 = e2 = tp = ta = n = 0
    for i in ids:
        tr, tf, v, x = TR[i], TF[i], Vi[i], Tx[i]
        e1 += ma.cos(v, tr) > ma.cos(v, tf)              # E1: vídeo↔título
        e2 += ma.cos(x, tr) > ma.cos(x, tf)              # E2: transcript↔título
        d_vt = 1 - ma.cos(v, x)                          # arista vídeo↔transcript (igual real/fake)
        pr = (1-ma.cos(tr, v)) + (1-ma.cos(tr, x)) + d_vt
        pf = (1-ma.cos(tf, v)) + (1-ma.cos(tf, x)) + d_vt
        tp += pr < pf                                    # triángulo real más pequeño = más coherente
        ta += _area3(tr, v, x) < _area3(tf, v, x)
        n += 1
    fv_2afc.append({'model': ma.MODEL_DISPLAY[model], 'N': n,
                    'E1 (t-v)': round(e1/n, 3), 'E2 (t-tr)': round(e2/n, 3),
                    'tri_perim': round(tp/n, 3), 'tri_area': round(ta/n, 3)})
    print(f"[2AFC] {ma.MODEL_DISPLAY[model]:13s} N={n}  E1={e1/n:.3f}  E2={e2/n:.3f}  "
          f"tri_perim={tp/n:.3f}  tri_area={ta/n:.3f}")
df_fv_2afc = pd.DataFrame(fv_2afc).set_index('model')
display(df_fv_2afc)

In [ ]:
N_REPS_FV, TRAIN_FV = 50, 0.7   # usa _l2 / _area3 de la celda B (ejecútala antes)

def _fit_thr_high(s, l):
    o = np.argsort(s, kind='mergesort'); ss, ls = s[o], l[o]
    n, nf = len(s), int(l.sum()); cum = np.concatenate([[0], np.cumsum(ls)])
    correct = nf + np.arange(n + 1) - 2 * cum; bk = int(np.argmax(correct))
    if bk == 0: return ss[0] - 1e-9
    if bk == n: return ss[-1] + 1e-9
    return (ss[bk-1] + ss[bk]) / 2

fv_thr = []
for model in FV_MODELS:
    mods = ma.available('FakeVV', model).get('modalities', {})
    if not {'text_real', 'text_fake', 'video', 'transcript'} <= set(mods):
        print(f"[skip] FakeVV thr {model}: sin transcript"); continue
    conn = ma.connect(ma.db_path('FakeVV', model))
    TR, TF = ma.load_globals(conn, 'text_real'), ma.load_globals(conn, 'text_fake')
    Vi, Tx = ma.load_globals(conn, 'video'), ma.load_globals(conn, 'transcript')
    conn.close()
    vids = sorted(set(TR) & set(TF) & set(Vi) & set(Tx)); N = len(vids)
    S = {k: [] for k in ['E1', 'E1+E2', 'tri_perim', 'tri_area']}; y, grp = [], []
    for gi, i in enumerate(vids):
        for title, lab in [(TR[i], 0), (TF[i], 1)]:
            d_tv, d_tt, d_vt = 1-ma.cos(title, Vi[i]), 1-ma.cos(title, Tx[i]), 1-ma.cos(Vi[i], Tx[i])
            S['E1'].append(-ma.cos(Vi[i], title)); S['E1+E2'].append(d_tv + d_tt)
            S['tri_perim'].append(d_tv + d_tt + d_vt); S['tri_area'].append(_area3(title, Vi[i], Tx[i]))
            y.append(lab); grp.append(gi)
    S = {k: np.array(v) for k, v in S.items()}; y = np.array(y); grp = np.array(grp)
    res = {k: [] for k in S}
    for rep in range(N_REPS_FV):
        rng = np.random.default_rng(1000 + rep); perm = rng.permutation(N); ktr = int(round(TRAIN_FV * N))
        trv = set(perm[:ktr].tolist()); tr = np.array([g in trv for g in grp]); te = ~tr
        for k in S:
            thr = _fit_thr_high(S[k][tr], y[tr])
            res[k].append(((S[k][te] >= thr).astype(int) == y[te]).mean())
    row = {'model': ma.MODEL_DISPLAY[model], 'N': N, **{k: round(float(np.mean(v)), 3) for k, v in res.items()}}
    fv_thr.append(row)
    print(f"[thr] {ma.MODEL_DISPLAY[model]:13s} N={N}  " + "  ".join(f"{k}={np.mean(v):.3f}" for k, v in res.items()))
df_fv_thr = pd.DataFrame(fv_thr).set_index('model')
display(df_fv_thr)

### E1c — Diagnóstico: ¿por qué paired ≈ 0.9 pero umbral ≈ 0.6?


In [ ]:
# ── E1c: ¿por qué paired ~0.9 pero umbral ~0.6? ──────────────────────────────
# Diagnóstico: descomposición de varianza + calibración por vídeo.
N_REPS_C, TRAIN_FRAC_C = 50, 0.7

def _l2rows(M):
    n = np.linalg.norm(M, axis=1, keepdims=True)
    n[n == 0] = 1.0
    return M / n

def fit_threshold_c(scores, labels):
    '''Umbral que maximiza accuracy en train para la regla `fake si score < thr`.'''
    s = np.asarray(scores, float); l = np.asarray(labels, int)
    order = np.argsort(s, kind='mergesort')
    s_sorted, l_sorted = s[order], l[order]
    n, n_fake = len(s), int(l.sum())
    fake_in_k = np.concatenate([[0], np.cumsum(l_sorted)])
    k = np.arange(n + 1)
    correct = fake_in_k + ((n - n_fake) - (k - fake_in_k))
    best_k = int(np.argmax(correct))
    if best_k == 0:   return float(s_sorted[0] - 1e-9)
    if best_k == n:   return float(s_sorted[-1] + 1e-9)
    return float((s_sorted[best_k - 1] + s_sorted[best_k]) / 2)

e1c_rows = []
for model in ma.MODELS_BY_DATASET['FakeVV']:
    info = ma.available('FakeVV', model)
    mods = info.get('modalities', {}) if info.get('exists') else {}
    if not (info.get('exists') and 'video' in mods and 'text_real' in mods and 'text_fake' in mods):
        continue
    conn = ma.connect(ma.db_path('FakeVV', model))
    vid  = ma.load_globals(conn, 'video')
    t_rl = ma.load_globals(conn, 'text_real')
    t_fk = ma.load_globals(conn, 'text_fake')
    conn.close()

    ids = sorted(set(vid) & set(t_rl) & set(t_fk))
    n = len(ids)
    V  = _l2rows(np.stack([vid[r]  for r in ids]).astype(np.float64))
    Tr = _l2rows(np.stack([t_rl[r] for r in ids]).astype(np.float64))
    Tf = _l2rows(np.stack([t_fk[r] for r in ids]).astype(np.float64))

    C      = V @ Tr.T                 # cos(V_i, title_real_j) — corpus de títulos
    cos_r  = np.diag(C).copy()        # cos(V_i, su título real)
    cos_f  = np.einsum('ij,ij->i', V, Tf)   # cos(V_i, su título fake)

    # 1) Descomposición: hueco intra-vídeo vs variación entre vídeos
    gap    = cos_r - cos_f
    level  = (cos_r + cos_f) / 2
    paired = float((gap > 0).mean())
    ratio  = float(gap.mean() / level.std())
    corr_rf = float(np.corrcoef(cos_r, cos_f)[0, 1])   # nivel compartido del par

    # 2) Calibración por vídeo: score = cos(V_i, t) − media_j∈train cos(V_i, título_real_j)
    accs_raw, accs_cal = [], []
    for rep in range(N_REPS_C):
        rng = np.random.default_rng(1000 + rep)
        perm = rng.permutation(n)
        k = int(round(TRAIN_FRAC_C * n))
        tr_idx, te_idx = perm[:k], perm[k:]

        mask = np.zeros(n, bool); mask[tr_idx] = True
        mu = (C[:, mask].sum(axis=1) - np.where(mask, cos_r, 0.0)) / (mask.sum() - mask.astype(int))

        for scores_r, scores_f, accs in [(cos_r, cos_f, accs_raw),
                                         (cos_r - mu, cos_f - mu, accs_cal)]:
            thr = fit_threshold_c(np.concatenate([scores_r[tr_idx], scores_f[tr_idx]]),
                                  [0] * len(tr_idx) + [1] * len(tr_idx))
            te_s = np.concatenate([scores_r[te_idx], scores_f[te_idx]])
            te_l = np.array([0] * len(te_idx) + [1] * len(te_idx))
            accs.append(float(((te_s < thr).astype(int) == te_l).mean()))

    row = {"model": model, "paired_acc": round(paired, 4),
           "gap_mean": round(float(gap.mean()), 4),
           "gap_std": round(float(gap.std()), 4),
           "between_video_std": round(float(level.std()), 4),
           "gap_over_between": round(ratio, 3),
           "corr_real_fake": round(corr_rf, 3),
           "thr_acc_raw": round(float(np.mean(accs_raw)), 4),
           "thr_acc_calibrated": round(float(np.mean(accs_cal)), 4),
           "thr_acc_cal_std": round(float(np.std(accs_cal)), 4)}
    e1c_rows.append(row)
    if model == 'ge2':
        _e1c_plot = {"cos_r": cos_r.copy(), "cos_f": cos_f.copy(), "gap": gap.copy()}
    print(f"[E1c] {ma.MODEL_DISPLAY[model]:13s} hueco intra={row['gap_mean']:+.4f}  "
          f"σ entre-vídeos={row['between_video_std']:.4f}  ratio={row['gap_over_between']:.2f}  "
          f"corr(real,fake)={corr_rf:.2f}  "
          f"acc umbral: cruda={row['thr_acc_raw']:.3f} → calibrada={row['thr_acc_calibrated']:.3f}")

df_e1c = pd.DataFrame(e1c_rows)
print()
print(df_e1c.to_string(index=False))

# ── Figura diagnóstica (GE2): por qué el umbral no puede ─────────────────────
if '_e1c_plot' in globals() or '_e1c_plot' in dir():
    cr, cf, gp = _e1c_plot["cos_r"], _e1c_plot["cos_f"], _e1c_plot["gap"]
    with plt.rc_context(ma._RC):
        fig, axes = plt.subplots(1, 3, figsize=(14, 3.8), constrained_layout=True)

        bins = np.linspace(min(cf.min(), cr.min()), max(cf.max(), cr.max()), 40)
        axes[0].hist(cr, bins=bins, alpha=0.6, color="#2E75B6", label="cos(V, real title)")
        axes[0].hist(cf, bins=bins, alpha=0.6, color="#E74C3C", label="cos(V, fake title)")
        axes[0].set_xlabel("cosine"); axes[0].set_ylabel("videos")
        axes[0].set_title("Pooled distributions: massive overlap")
        axes[0].legend(fontsize=8)

        axes[1].scatter(cr, cf, s=8, alpha=0.4, color="#4472C4")
        lims = [min(cr.min(), cf.min()), max(cr.max(), cf.max())]
        axes[1].plot(lims, lims, "--", color="#E74C3C", lw=1.2, label="y = x")
        axes[1].set_xlabel("cos(V, real title)"); axes[1].set_ylabel("cos(V, fake title)")
        axes[1].set_title(f"Shared per-video level (r = {np.corrcoef(cr, cf)[0,1]:.2f})")
        axes[1].legend(fontsize=8)

        axes[2].hist(gp, bins=40, color="#70AD47", alpha=0.8)
        axes[2].axvline(0, color="#E74C3C", ls="--", lw=1.2)
        axes[2].set_xlabel("within-video gap  cos_real − cos_fake"); axes[2].set_ylabel("videos")
        axes[2].set_title(f"Within-video gap: {100*(gp>0).mean():.0f}% > 0, but tiny")

        fig.suptitle("FakeVV E1c (GE2) — why paired ranking ≈ 0.9 but threshold ≈ 0.63", fontsize=12)
        fig.savefig(str(ma.FIG_DIR / "fakevv_e1c_diagnosis.png"), dpi=150, bbox_inches="tight")
        plt.show(); plt.close('all')

In [ ]:
# ── E1d: ¿un umbral condicionado al vídeo recupera la señal? ──────────────────
# raw → calibrado (resta media por-vídeo) → z-score (resta media y divide por σ por-vídeo).
# La referencia por-vídeo (μ_V, σ_V) se estima del corpus de títulos reales en TRAIN (leave-one-out).
N_REPS_D, TRAIN_FRAC_D = 50, 0.7

def _l2rows_d(M):
    n = np.linalg.norm(M, axis=1, keepdims=True); n[n == 0] = 1.0
    return M / n

def fit_threshold_d(scores, labels):
    s = np.asarray(scores, float); l = np.asarray(labels, int)
    order = np.argsort(s, kind='mergesort')
    s_s, l_s = s[order], l[order]
    n, nf = len(s), int(l.sum())
    fake_in_k = np.concatenate([[0], np.cumsum(l_s)])
    k = np.arange(n + 1)
    correct = fake_in_k + ((n - nf) - (k - fake_in_k))
    bk = int(np.argmax(correct))
    if bk == 0: return float(s_s[0] - 1e-9)
    if bk == n: return float(s_s[-1] + 1e-9)
    return float((s_s[bk - 1] + s_s[bk]) / 2)

e1d_rows = []
for model in ma.MODELS_BY_DATASET['FakeVV']:
    info = ma.available('FakeVV', model)
    mods = info.get('modalities', {}) if info.get('exists') else {}
    if not (info.get('exists') and 'video' in mods and 'text_real' in mods and 'text_fake' in mods):
        continue
    conn = ma.connect(ma.db_path('FakeVV', model))
    vid, t_rl, t_fk = ma.load_globals(conn, 'video'), ma.load_globals(conn, 'text_real'), ma.load_globals(conn, 'text_fake')
    conn.close()

    ids = sorted(set(vid) & set(t_rl) & set(t_fk)); n = len(ids)
    V  = _l2rows_d(np.stack([vid[r]  for r in ids]).astype(np.float64))
    Tr = _l2rows_d(np.stack([t_rl[r] for r in ids]).astype(np.float64))
    Tf = _l2rows_d(np.stack([t_fk[r] for r in ids]).astype(np.float64))

    C     = V @ Tr.T                          # cos(V_i, título_real_j) — corpus de referencia
    cos_r = np.diag(C).copy()
    cos_f = np.einsum('ij,ij->i', V, Tf)
    paired = float((cos_r > cos_f).mean())

    accs = {'raw': [], 'cal': [], 'z': []}
    for rep in range(N_REPS_D):
        rng = np.random.default_rng(2000 + rep)
        perm = rng.permutation(n); k = int(round(TRAIN_FRAC_D * n))
        tr_idx, te_idx = perm[:k], perm[k:]
        mask = np.zeros(n, bool); mask[tr_idx] = True

        # μ_V y σ_V por vídeo, sobre títulos reales de TRAIN (leave-one-out para los de train)
        cnt   = mask.sum() - mask.astype(int)                                  # |trai
        s1    = C[:, mask].sum(axis=1) - np.where(mask, cos_r, 0.0)
        s2    = (C[:, mask] ** 2).sum(axis=1) - np.where(mask, cos_r ** 2, 0.0)
        mu    = s1 / cnt
        var   = np.maximum(s2 / cnt - mu ** 2, 1e-12)
        sigma = np.sqrt(var)

        variants = {
            'raw': (cos_r,            cos_f),
            'cal': (cos_r - mu,       cos_f - mu),
            'z':   ((cos_r - mu)/sigma, (cos_f - mu)/sigma),
        }
        for key, (sr, sf) in variants.items():
            thr = fit_threshold_d(np.concatenate([sr[tr_idx], sf[tr_idx]]),
                                  [0]*len(tr_idx) + [1]*len(tr_idx))
            te_s = np.concatenate([sr[te_idx], sf[te_idx]])
            te_l = np.array([0]*len(te_idx) + [1]*len(te_idx))
            accs[key].append(float(((te_s < thr).astype(int) == te_l).mean()))

    row = {'model': model, 'paired': round(paired, 4),
           'thr_raw':        round(float(np.mean(accs['raw'])), 4),
           'thr_calibrated': round(float(np.mean(accs['cal'])), 4),
           'thr_zscore':     round(float(np.mean(accs['z'])),   4),
           'zscore_std':     round(float(np.std(accs['z'])),    4)}
    e1d_rows.append(row)
print(f"[E1d] {ma.MODEL_DISPLAY[model]:13s} paired={row['paired']:.3f}  "
      f"raw={row['thr_raw']:.3f} → cal={row['thr_calibrated']:.3f} → z={row['thr_zscore']:.3f}")

df_e1d = pd.DataFrame(e1d_rows)
print(); print(df_e1d.to_string(index=False))

# ── Figura: escalera paired → raw → calibrado → z-score ──────────────────────
groups = {
    'Paired (ranking)':       {r['model']: r['paired']         for r in e1d_rows},
    'Threshold raw':          {r['model']: r['thr_raw']        for r in e1d_rows},
    'Threshold +per-video μ': {r['model']: r['thr_calibrated'] for r in e1d_rows},
    'Threshold +per-video z': {r['model']: r['thr_zscore']     for r in e1d_rows},
}
fig = ma.grouped_bar(groups, ylabel='Accuracy',
                     title='FakeVV — Can a per-video-conditioned threshold recover the paired signal?',
                     fname='fakevv_e1d_pervideo_threshold.png', baseline=0.5, pct=True)
plt.show(); plt.close('all')

In [ ]:
# FakeVV - tres regímenes por arista (Paired 2AFC, AUC, Threshold). Autocontenida.
# fakeness por ítem (alto = fake): aristas = -cos, triángulo = +area_true. Pares 50/50 -> chance 0.5.
N_REPS_FVR, TRAIN_FVR = 50, 0.7
def _l2(v): n = np.linalg.norm(v); return v / n if n > 1e-12 else v
def _area3(a, b, c):
    a, b, c = _l2(a), _l2(b), _l2(c); u, v = b - a, c - a
    return 0.5 * np.sqrt(max(float(u@u*(v@v) - (u@v)**2), 0.0))
def _fit_thr_high(s, l):
    o = np.argsort(s, kind='mergesort'); ss, ls = s[o], l[o]
    n, nf = len(s), int(l.sum()); cum = np.concatenate([[0], np.cumsum(ls)])
    correct = nf + np.arange(n + 1) - 2 * cum; bk = int(np.argmax(correct))
    if bk == 0: return ss[0] - 1e-9
    if bk == n: return ss[-1] + 1e-9
    return (ss[bk-1] + ss[bk]) / 2
def _thr_grouped(score, label, groups, seed, nrep=N_REPS_FVR, frac=TRAIN_FVR):
    s = np.asarray(score, float); y = np.asarray(label, int); g = np.asarray(groups)
    uniq = np.unique(g); accs = []
    for rep in range(nrep):
        rng = np.random.default_rng(seed + rep)
        trv = set(rng.permutation(uniq)[:int(round(frac*len(uniq)))].tolist())
        tr = np.array([x in trv for x in g]); te = ~tr
        thr = _fit_thr_high(s[tr], y[tr])
        accs.append(((s[te] >= thr).astype(int) == y[te]).mean())
    return float(np.mean(accs))

fv_reg = []
for model in ma.MODELS_BY_DATASET['FakeVV']:
    mods = ma.available('FakeVV', model).get('modalities', {})
    if not {'text_real', 'text_fake', 'video', 'transcript'} <= set(mods):
        print(f"[skip] FakeVV {model}: falta transcript (WAVE) o aristas"); continue
    conn = ma.connect(ma.db_path('FakeVV', model))
    TR, TF = ma.load_globals(conn, 'text_real'), ma.load_globals(conn, 'text_fake')
    Vi, Tx = ma.load_globals(conn, 'video'), ma.load_globals(conn, 'transcript')
    conn.close()
    ids = sorted(set(TR) & set(TF) & set(Vi) & set(Tx)); N = len(ids)
    edges = {
        'E1 video-title':      ([-ma.cos(Vi[i], TR[i]) for i in ids], [-ma.cos(Vi[i], TF[i]) for i in ids]),
        'E2 transcript-title': ([-ma.cos(Tx[i], TR[i]) for i in ids], [-ma.cos(Tx[i], TF[i]) for i in ids]),
        'Triangle title-v-tr': ([_area3(TR[i], Vi[i], Tx[i]) for i in ids], [_area3(TF[i], Vi[i], Tx[i]) for i in ids]),
    }
    for exp, (rf, ff) in edges.items():
        rf = np.array(rf); ff = np.array(ff)
        paired = float((ff > rf).mean())
        sc = np.concatenate([rf, ff]); y = np.array([0]*N + [1]*N); grp = np.array(list(range(N))*2)
        auc = float(ma.auc_fake(sc, y)); thr = _thr_grouped(sc, y, grp, seed=5000)
        fv_reg.append(dict(dataset='FakeVV', exp=exp, model=model, N=N,
                           paired=round(paired,3), auc=round(auc,3), thr=round(thr,3)))
        print(f"[FV] {ma.MODEL_DISPLAY[model]:13s} {exp:22s} N={N:4d}  2AFC={paired:.3f}  AUC={auc:.3f}  thr={thr:.3f}")

df_fv_reg = pd.DataFrame(fv_reg)
for metric in ['paired', 'auc', 'thr']:
    print(f"\n== FakeVV {metric} =="); display(df_fv_reg.pivot_table(index='exp', columns='model', values=metric).round(3))


### Conclusión — Bloque 1: FakeVV (entity‑swap título↔vídeo)
#### Contexto y alcance
#### 1. ¿Existe señal? Discriminación 2AFC
#### 2. ¿De qué depende? Por categoría de entidad
#### 3. El golpe de realidad: del ranking al umbral
#### 4. ¿Por qué? El diagnóstico
#### 5. ¿Y un clasificador aprendido? Probes sobre los embeddings
#### 6. Con qué me quedo
#### 7. Hacia el siguiente bloque (GroundLie360)


## 5. Dataset B — GroundLie360 (taxonomía de mentiras E1–E6b)


In [ ]:
# Pre-cargar labels de GroundLie360 para todos los modelos disponibles
GL_MODELS = ma.MODELS_BY_DATASET['GroundLie360']
gl_conns = {}
gl_labels = {}
for model in GL_MODELS:
    info = ma.available('GroundLie360', model)
    if not info.get('exists'):
        print(f"[skip] GroundLie360 {model}: DB not found")
        continue
    conn = ma.connect(ma.db_path('GroundLie360', model))
    try:
        lab = ma.load_labels(conn)
        gl_labels[model] = lab
        gl_conns[model] = conn
        print(f"  {ma.MODEL_DISPLAY[model]:15s}  N_labels={len(lab)}  "
              f"fake={lab.binary_label.sum()}  real={(lab.binary_label==0).sum()}")
    except Exception as ex:
        print(f"[skip] GroundLie360 {model} labels error: {ex}")
        conn.close()


### E1 — cos(video, title): real vs fake


In [ ]:
import warnings
e1_gl = {}

for model in GL_MODELS:
    if model not in gl_labels:
        print(f"[skip] GL E1 {model}: no labels")
        continue
    info = ma.available('GroundLie360', model)
    mods = info.get('modalities', {})
    if 'video' not in mods or 'text_title' not in mods:
        print(f"[skip] GL E1 {model}: video or text_title missing (mods={list(mods.keys())})")
        continue
    conn = gl_conns[model]
    vid = ma.load_globals(conn, 'video')
    tit = ma.load_globals(conn, 'text_title')
    lab = gl_labels[model]

    recs = []
    for rid in set(vid) & set(tit) & set(lab.index):
        recs.append({
            'raw_id': rid,
            'cos_vt': ma.cos(vid[rid], tit[rid]),
            'binary_label': int(lab.loc[rid, 'binary_label']),
            'false_title': int(lab.loc[rid, 'false_title']),
        })
    df = pd.DataFrame(recs)
    e1_gl[model] = df

    real_m = df[df.binary_label==0].cos_vt.mean()
    fake_m = df[df.binary_label==1].cos_vt.mean()
    auc    = ma.auc_fake(-df.cos_vt, df.binary_label)
    print(f"[E1] {ma.MODEL_DISPLAY[model]:15s}  N={len(df):4d}  "
          f"cos_real={real_m:.4f}  cos_fake={fake_m:.4f}  Δ={real_m-fake_m:+.4f}  AUC(-cos)={auc:.3f}")


In [ ]:
# Figures for E1
if e1_gl:
    # AUC by model
    auc_vals = {m: ma.auc_fake(-df.cos_vt, df.binary_label) for m, df in e1_gl.items()}
    fig = ma.grouped_bar(auc_vals,
                         ylabel='AUC (score = -cos)',
                         title='GroundLie360 E1 — AUC(-cos_title_video) by model',
                         fname='gl360_e1_auc.png', baseline=0.5)
    plt.show(); plt.close('all')

    # Distribution real vs fake (qw8b if available, else first)
    ref_m = 'qw8b' if 'qw8b' in e1_gl else next(iter(e1_gl))
    df_ref = e1_gl[ref_m]
    fig = ma.dist_real_vs_fake(
        df_ref[df_ref.binary_label==0].cos_vt,
        df_ref[df_ref.binary_label==1].cos_vt,
        xlabel='cos(video, title)',
        title=f'GroundLie360 E1 — cos distribution ({ma.MODEL_DISPLAY[ref_m]})',
        fname='gl360_e1_dist.png',
    )
    plt.show(); plt.close('all')


In [ ]:
from sklearn.metrics import roc_auc_score

# ── GroundLie E1 by fake type — AUC(-cos) title↔video, with N per type ────────
GL_TYPES = ['false_title', 'false_speech', 'temporal_edit', 'cgi', 'contradictory', 'unsupported']

# Labels (same across models) — read from the qw2b DB
_lc = ma.connect(ma.db_path('GroundLie360', 'qw2b'))
_lr = _lc.execute(f"SELECT raw_id, binary_label, {','.join(GL_TYPES)} FROM groundlie_labels").fetchall()
_lc.close()
gl_lab  = {r[0]: {'bin': r[1], **{t: r[2 + i] for i, t in enumerate(GL_TYPES)}} for r in _lr}
gl_real = {i for i, d in gl_lab.items() if d['bin'] == 0}
n_by_type = {'ALL fake': sum(d['bin'] == 1 for d in gl_lab.values()),
             **{t: sum(d.get(t) == 1 for d in gl_lab.values()) for t in GL_TYPES}}

def _auc_negcos(cos_by_id, pos_ids):
    ids = [i for i in cos_by_id if i in pos_ids or i in gl_real]
    y = np.array([1 if i in pos_ids else 0 for i in ids])
    if y.sum() == 0 or y.sum() == len(y):
        return np.nan
    return roc_auc_score(y, np.array([-cos_by_id[i] for i in ids]))   # -cos: higher = more "fake"

auc = {t: {'N': n_by_type[t]} for t in ['ALL fake'] + GL_TYPES}
for model in ma.MODELS_BY_DATASET['GroundLie360']:
    info = ma.available('GroundLie360', model)
    mods = info.get('modalities', {}) if info.get('exists') else {}
    if not (info.get('exists') and 'video' in mods and 'text_title' in mods):
        for t in auc:
            auc[t][model] = np.nan
        continue
    conn = ma.connect(ma.db_path('GroundLie360', model))
    vid, tit = ma.load_globals(conn, 'video'), ma.load_globals(conn, 'text_title')
    conn.close()
    cos = {i: ma.cos(vid[i], tit[i]) for i in vid if i in tit}
    for t in ['ALL fake'] + GL_TYPES:
        pos = ({i for i in cos if gl_lab.get(i, {}).get('bin') == 1} if t == 'ALL fake'
               else {i for i in cos if gl_lab.get(i, {}).get(t) == 1})
        auc[t][model] = round(_auc_negcos(cos, pos), 3)

model_cols = list(ma.MODELS_BY_DATASET['GroundLie360'])
df_gl_auc = pd.DataFrame(auc).T[['N'] + model_cols]
df_gl_auc.columns = ['N'] + [ma.MODEL_DISPLAY[m] for m in model_cols]

print(f"GroundLie360 — AUC(-cos) title↔video by fake type   (real N={len(gl_real)})")
print("AUC > 0.5 = expected direction (fake misaligns) | < 0.5 = inverted (fake aligns more)\n")
display(df_gl_auc)

# ── Figure: AUC by fake type and model, chance line at 0.5 ────────────────────
groups = {t: {m: auc[t][m] for m in model_cols} for t in GL_TYPES}
fig = ma.grouped_bar(groups, ylabel='AUC(-cos)  title↔video',
                     title='GroundLie360 — Title↔video detection by fake type (chance = 0.5)',
                     fname='groundlie_auc_by_faketype.png', baseline=0.5, pct=False)
plt.show(); plt.close('all')

### E2 — ventana textual (False Speech)


In [ ]:
e2_gl = {}

for model in GL_MODELS:
    if model not in gl_labels:
        print(f"[skip] GL E2 {model}: no labels")
        continue
    info = ma.available('GroundLie360', model)
    mods = info.get('modalities', {})
    segs = info.get('segments', {})
    if 'text_title' not in mods:
        print(f"[skip] GL E2 {model}: text_title missing")
        continue

    conn = gl_conns[model]
    if model == 'ge2':
        seg_info = ma.available('GroundLie360', 'ge2')
        # GE2 segments in separate DB
        if not ma.GE2_GL360_SEGMENTS_DB.exists():
            print(f"[skip] GL E2 ge2: segments DB not found at {ma.GE2_GL360_SEGMENTS_DB}")
            continue
        seg_conn = ma.connect(ma.GE2_GL360_SEGMENTS_DB)
    else:
        seg_conn = conn

    try:
        fake_segs  = ma.load_segments(seg_conn, 'window_fake',  'transcript')
        clean_segs = ma.load_segments(seg_conn, 'window_clean', 'transcript')
    except Exception as ex:
        print(f"[skip] GL E2 {model}: segment load error: {ex}")
        if model == 'ge2': seg_conn.close()
        continue

    tit = ma.load_globals(conn, 'text_title')

    if not fake_segs or not clean_segs:
        print(f"[skip] GL E2 {model}: no window segments found")
        if model == 'ge2': seg_conn.close()
        continue

    recs = []
    for rid in set(fake_segs) & set(clean_segs) & set(tit):
        cf = ma.cos(fake_segs[rid][0]['vector'],  tit[rid])
        cc = ma.cos(clean_segs[rid][0]['vector'], tit[rid])
        recs.append({'raw_id': rid, 'cos_fake': cf, 'cos_clean': cc, 'delta': cc - cf})

    if model == 'ge2':
        seg_conn.close()

    df = pd.DataFrame(recs)
    e2_gl[model] = df
    print(f"[E2] {ma.MODEL_DISPLAY[model]:15s}  N={len(df):3d}  "
          f"Δ medio={df.delta.mean():+.4f}  %Δ>0={100*(df.delta>0).mean():.0f}%")

if not e2_gl:
    print("[skip] GL E2: no models had window segments — run E2 embedding jobs first")


### E3 — coherencia entre escenas adyacentes (Temporal Edit)


In [ ]:
# E3 — PREGUNTA: ¿en promedio, los vídeos con edición temporal son MENOS coherentes
#                entre escenas que los no editados?  (detección a nivel vídeo)

# CÓMO: por vídeo, media del coseno entre TODAS las escenas adyacentes (mean_adj_cos);
#       luego se compara la media entre editados (TE1) y no editados (TE0).


e3_gl = {}
for model in GL_MODELS:
    if model == 'wave':
        print(f"[skip] GL E3 wave: WAVE scene embeddings invalidated by <image> token bug "
              f"(adj-cos=1.0). Excluded until regeneration complete.")
        continue
    if model not in gl_labels:
        print(f"[skip] GL E3 {model}: no labels")
        continue
    info = ma.available('GroundLie360', model)
    segs = info.get('segments', {})

    conn = gl_conns[model]
    if model == 'ge2':
        if not ma.GE2_GL360_SEGMENTS_DB.exists():
            print(f"[skip] GL E3 ge2: segments DB not found")
            continue
        seg_conn = ma.connect(ma.GE2_GL360_SEGMENTS_DB)
    else:
        seg_conn = conn

    try:
        scenes = ma.load_segments(seg_conn, 'scene', 'video')
    except Exception as ex:
        print(f"[skip] GL E3 {model}: scene segment load error: {ex}")
        if model == 'ge2': seg_conn.close()
        continue

    if model == 'ge2':
        seg_conn.close()

    lab = gl_labels[model]
    recs = []
    for rid, scene_list in scenes.items():
        if len(scene_list) < 2 or rid not in lab.index:
            continue
        adj = [ma.cos(scene_list[i]['vector'], scene_list[i+1]['vector'])
               for i in range(len(scene_list)-1)]
        recs.append({
            'raw_id': rid,
            'mean_adj_cos': float(np.nanmean(adj)),
            'n_scenes': len(scene_list),
            'temporal_edit': int(lab.loc[rid, 'temporal_edit']),
            'binary_label': int(lab.loc[rid, 'binary_label']),
        })

    if not recs:
        print(f"[skip] GL E3 {model}: no scene segments with ≥2 scenes")
        continue

    df = pd.DataFrame(recs)
    e3_gl[model] = df
    te0 = df[df.temporal_edit==0].mean_adj_cos.mean()
    te1 = df[df.temporal_edit==1].mean_adj_cos.mean()
    auc = ma.auc_fake(-df.mean_adj_cos, df.temporal_edit)
    print(f"[E3] {ma.MODEL_DISPLAY[model]:15s}  N={len(df):4d}  "
          f"adj-cos TE0={te0:.4f}  TE1={te1:.4f}  Δ(TE0-TE1)={te0-te1:+.4f}  "
          f"AUC(-cos|TE)={auc:.3f}")

if not e3_gl:
    print("[info] GL E3: no models produced scene-coherence results in this run")


In [ ]:
# E3 scope — PREGUNTA: ¿qué fracción de las ediciones temporales puede VER este método?
# CLAVE: la coherencia escena↔escena solo ve cortes INTERNOS (entre dos escenas).
#        Los recortes de EXTREMO (inicio/fin) no tienen escena vecina -> invisibles.
# mask = [START, transición_1, …, transición_{n-1}, END]  (longitud n+1)


import json
from sklearn.metrics import roc_auc_score  # (lo usa E3b)

gl_index = json.loads((ma.REPO_ROOT / "experiments/GroundLie360/groundlie_index_filtered.json").read_text(encoding="utf-8"))
gl_meta  = {e['raw_id']: e for e in gl_index}

GL_E3_MODELS = ['qw2b', 'qw8b', 'ge2']   # añade 'wave' tras re-pull de la DB GroundLie (la local estaba corrupta)

def _transitions(scene_dicts):
    v = [s['vector'] for s in scene_dicts]
    return np.array([ma.cos(v[t], v[t+1]) for t in range(len(v)-1)])

def _scenes_for(model):
    if model == 'ge2':
        if not ma.GE2_GL360_SEGMENTS_DB.exists():
            return None
        sc = ma.connect(ma.GE2_GL360_SEGMENTS_DB)
    else:
        sc = ma.connect(ma.db_path('GroundLie360', model))
    out = ma.load_segments(sc, 'scene', 'video'); sc.close()
    return out

# ── Scope: ¿cuántos temporal edits son internos (transiciones) vs extremos (start/end)? ──
te = [e for e in gl_index if e.get('temporal_edit') == 1 and len(e.get('temporal_edit_mask', [])) >= 2]
sc_int = sc_bnd = sc_both = 0
for e in te:
    m = e['temporal_edit_mask']; n = len(m) - 1
    hi = any(m[i] == 1 for i in range(1, n)); hb = (m[0] == 1) or (m[n] == 1)
    sc_both += hi and hb; sc_int += hi and not hb; sc_bnd += hb and not hi
print(f"Temporal-edit scope: internal-only={sc_int}, boundary-only={sc_bnd}, both={sc_both}, total={len(te)}")
print("(scene↔scene coherence solo ve transiciones INTERNAS; los edits de extremos quedan fuera de alcance)\n")


In [ ]:
# E3a — PREGUNTA: dado un vídeo que SABEMOS editado, ¿es el corte marcado el de
#                 MENOR coherencia de su vídeo?  (LOCALIZACIÓN, no detección)

# CÓMO: ordeno las transiciones por coseno (menor→mayor) y miro la posición del corte
#       marcado (temporal_edit_mask).  mask[i] ↔ transición i-1 (confirmado en el paper).
#     hit@1        = fracción donde el corte marcado es el MÍNIMO (posición 0)
#     mean_rank    = posición normalizada media del marcado (0=el más bajo, 0.5=azar)
#     chance_hit@1 = azar (≈ nº marcados / nº transiciones)


e3a_rows = []
for model in GL_E3_MODELS:
    scenes = _scenes_for(model)
    if scenes is None:
        print(f"[skip] {model}: no segments DB"); continue
    hits = 0; ranks = []; chance = []; n_used = 0
    for rid, scn in scenes.items():
        e = gl_meta.get(rid)
        if not e or e.get('temporal_edit') != 1:
            continue
        m = e.get('temporal_edit_mask', []); n = len(scn)
        if n != len(e.get('scene_frames', [])) or len(m) != n + 1 or n < 3:
            continue                                   # alineamiento garantizado + ≥2 transiciones
        internal = [i - 1 for i in range(1, n) if m[i] == 1]   # mask[i] -> transición i-1
        if not internal:
            continue                                   # edit de extremos: fuera de alcance
        coh = _transitions(scn)
        order = list(np.argsort(coh))                  # order[0] = transición de MENOR coherencia
        best = min(order.index(t) for t in internal)
        ranks.append(best / (len(coh) - 1)); chance.append(len(internal) / len(coh))
        hits += (best == 0); n_used += 1
    if n_used:
        e3a_rows.append({'model': ma.MODEL_DISPLAY[model], 'N': n_used,
                         'hit@1': round(hits / n_used, 3),
                         'mean_rank_norm': round(float(np.mean(ranks)), 3),
                         'chance_hit@1': round(float(np.mean(chance)), 3)})
        print(f"[E3a] {ma.MODEL_DISPLAY[model]:13s} N={n_used:3d}  hit@1={hits/n_used:.3f}  "
              f"mean_rank={np.mean(ranks):.3f}  chance hit@1={np.mean(chance):.3f}")

df_e3a = pd.DataFrame(e3a_rows)
display(df_e3a)

In [ ]:
# E3b — PREGUNTA: dado un vídeo CUALQUIERA, ¿podemos detectar si está editado
#                 buscando un corte anómalo?  (DETECCIÓN, no localización)
# CÓMO: por vídeo, estandarizo las coherencias de sus transiciones (z-score) y tomo
#       el "bajón" más profundo:  dip = -min(z).  Score alto = corte muy anómalo = editado.
#   • AUC new (min-z dip)  = detección con el corte más anómalo (esta hipótesis)
#   • AUC old (mean-cos)   = baseline: baja coherencia MEDIA = editado (E3 clásico)
# El z-score por vídeo quita el "nivel" propio (unos vídeos son naturalmente más saltarines).

e3b_rows = []
for model in GL_E3_MODELS:
    scenes = _scenes_for(model)
    if scenes is None:
        continue
    rows = []
    for rid, scn in scenes.items():
        e = gl_meta.get(rid)
        if not e or 'temporal_edit' not in e or len(scn) < 3:   # ≥3 escenas = ≥2 transiciones
            continue
        coh = _transitions(scn)
        sd = coh.std()
        dip = 0.0 if sd < 1e-9 else float(-((coh - coh.mean()) / sd).min())  # bajón estandarizado más profundo
        rows.append({'dip': dip, 'mean_cos': float(coh.mean()), 'te': int(e['temporal_edit'])})
    if not rows:
        continue
    df = pd.DataFrame(rows); y = df.te.values
    if y.sum() == 0 or y.sum() == len(y):
        continue
    auc_old = roc_auc_score(y, -df.mean_cos.values)   # E3 viejo: baja coherencia media = editado
    auc_new = roc_auc_score(y,  df.dip.values)        # E3 nuevo: bajón anómalo profundo = editado
    e3b_rows.append({'model': ma.MODEL_DISPLAY[model], 'N': len(df), 'edited': int(y.sum()),
                     'AUC old (mean-cos)': round(auc_old, 3),
                     'AUC new (min-z dip)': round(auc_new, 3)})
    print(f"[E3b] {ma.MODEL_DISPLAY[model]:13s} N={len(df):4d} edited={int(y.sum()):2d}  "
          f"AUC old={auc_old:.3f} -> new={auc_new:.3f}")

df_e3b = pd.DataFrame(e3b_rows)
display(df_e3b)

### E5 — Doble engaño: interacciones entre tipos de manipulación


In [ ]:
ref_model = 'qw8b' if 'qw8b' in e1_gl else (next(iter(e1_gl)) if e1_gl else None)

if ref_model:
    conn_ref = gl_conns[ref_model]
    lab_ref  = gl_labels[ref_model]
    df_ref   = e1_gl[ref_model].set_index('raw_id')
    # join with false_speech and temporal_edit
    extra_cols = ['false_speech', 'temporal_edit', 'cgi']
    available_extra = [c for c in extra_cols if c in lab_ref.columns]
    df_e5 = df_ref.join(lab_ref[available_extra])

    print(f"\n=== E5 — cos(titulo, video) medio [{ma.MODEL_DISPLAY[ref_model]}] ===")
    print("\nPor (false_title × false_speech):")
    if 'false_speech' in df_e5.columns:
        display(df_e5.groupby(['false_title', 'false_speech']).cos_vt.agg(['mean', 'count']).round(4))
    if 'temporal_edit' in df_e5.columns:
        print("\nPor (temporal_edit × false_title):")
        display(df_e5.groupby(['temporal_edit', 'false_title']).cos_vt.agg(['mean', 'count']).round(4))
else:
    print("[skip] GL E5: no model available for E5 analysis")


### E6 — Simplex V·A·T — WAVE only


In [ ]:
e6_wave = None

wave_info = ma.available('GroundLie360', 'wave')
wave_mods = wave_info.get('modalities', {}) if wave_info.get('exists') else {}

if not wave_info.get('exists'):
    print("[skip] GL E6: GroundLie360 WAVE DB not found")
elif 'audio' not in wave_mods:
    print(f"[skip] GL E6: WAVE audio embeddings missing (mods={list(wave_mods.keys())})")
elif 'video' not in wave_mods:
    print(f"[skip] GL E6: WAVE video embeddings missing — regeneration in progress")
else:
    if 'wave' in gl_conns:
        conn_wave = gl_conns['wave']
        lab_wave  = gl_labels.get('wave')
        V = ma.load_globals(conn_wave, 'video')
        A = ma.load_globals(conn_wave, 'audio')
        T = ma.load_globals(conn_wave, 'transcript')
        recs = []
        for rid in set(V) & set(A) & set(T) & (set(lab_wave.index) if lab_wave is not None else set(V)):
            d_va = 1 - ma.cos(V[rid], A[rid])
            d_vt = 1 - ma.cos(V[rid], T[rid])
            d_at = 1 - ma.cos(A[rid], T[rid])
            bl = int(lab_wave.loc[rid, 'binary_label']) if lab_wave is not None and rid in lab_wave.index else np.nan
            recs.append({'raw_id': rid, 'perim': d_va+d_vt+d_at, 'binary_label': bl})
        e6_wave = pd.DataFrame(recs)
        real_p = e6_wave[e6_wave.binary_label==0].perim.mean()
        fake_p = e6_wave[e6_wave.binary_label==1].perim.mean()
        auc6   = ma.auc_fake(e6_wave.perim, e6_wave.binary_label)
        print(f"[E6] WAVE V·A·T  N={len(e6_wave)}  "
              f"perim_real={real_p:.4f}  perim_fake={fake_p:.4f}  Δ={fake_p-real_p:+.4f}  "
              f"AUC(perim)={auc6:.3f}")
    else:
        print("[skip] GL E6: WAVE connection not available")


### E6b — Triángulo título·video·transcript (4 modelos)


In [ ]:
from sklearn.metrics import roc_auc_score

GL_TYPES = ['false_title', 'false_speech', 'temporal_edit', 'cgi', 'contradictory', 'unsupported']

def tri_area_true(a, b, c):
    """Área REAL del triángulo formado por las puntas de los 3 vectores (TRIANGLE, Cicchetti et al.).
    Normaliza a la esfera unidad y calcula 0.5·|u×v| en alta dimensión vía Gram."""
    a = a / (np.linalg.norm(a) + 1e-12)
    b = b / (np.linalg.norm(b) + 1e-12)
    c = c / (np.linalg.norm(c) + 1e-12)
    u, v = b - a, c - a
    cross2 = float(np.dot(u, u) * np.dot(v, v) - np.dot(u, v) ** 2)
    return 0.5 * np.sqrt(max(cross2, 0.0))

# ── E6b: triángulo título·vídeo·transcripción — perim vs area-Heron vs area-TRUE ──
e6b_gl = {}
for model in GL_MODELS:
    if model not in gl_labels:
        print(f"[skip] GL E6b {model}: no labels"); continue
    mods = ma.available('GroundLie360', model).get('modalities', {})
    missing = [m for m in ['text_title', 'video', 'transcript'] if m not in mods]
    if missing:
        print(f"[skip] GL E6b {model}: missing {missing}"); continue

    conn, lab = gl_conns[model], gl_labels[model]
    Ti = ma.load_globals(conn, 'text_title')
    Vi = ma.load_globals(conn, 'video')
    Tr = ma.load_globals(conn, 'transcript')

    recs = []
    for rid in set(Ti) & set(Vi) & set(Tr) & set(lab.index):
        d_tv  = 1 - ma.cos(Ti[rid], Vi[rid])
        d_tt  = 1 - ma.cos(Ti[rid], Tr[rid])
        d_vtr = 1 - ma.cos(Vi[rid], Tr[rid])
        recs.append({
            'raw_id': rid,
            'perim':      d_tv + d_tt + d_vtr,
            'area_heron': ma.tri_area(d_tv, d_tt, d_vtr),
            'area_true':  tri_area_true(Ti[rid], Vi[rid], Tr[rid]),
            'binary_label': int(lab.loc[rid, 'binary_label']),
            **{t: int(lab.loc[rid, t]) for t in GL_TYPES},
        })
    df = pd.DataFrame(recs)
    e6b_gl[model] = df
    a = {s: ma.auc_fake(df[s], df.binary_label) for s in ['perim', 'area_heron', 'area_true']}
    print(f"[E6b] {ma.MODEL_DISPLAY[model]:15s} N={len(df):4d}  "
          f"AUC perim={a['perim']:.3f}  area_heron={a['area_heron']:.3f}  area_true={a['area_true']:.3f}")

# ── Por tipo de fake (score = area_true) ──────────────────────────────────────
models = [m for m in GL_MODELS if m in e6b_gl]
ref = e6b_gl[models[0]]
n_used = {'ALL fake': int((ref.binary_label == 1).sum()),
          **{t: int((ref[t] == 1).sum()) for t in GL_TYPES}}
n_real = int((ref.binary_label == 0).sum())

def _auc_type(df, score, t):
    real = df[df.binary_label == 0]
    pos  = df[df.binary_label == 1] if t == 'ALL fake' else df[df[t] == 1]
    if len(pos) == 0:
        return np.nan
    y = np.r_[np.ones(len(pos)), np.zeros(len(real))]
    s = np.r_[pos[score].values, real[score].values]
    return roc_auc_score(y, s)

auc_t = {t: {'N': n_used[t]} for t in ['ALL fake'] + GL_TYPES}
for model in models:
    for t in ['ALL fake'] + GL_TYPES:
        auc_t[t][model] = round(_auc_type(e6b_gl[model], 'area_true', t), 3)

df_e6b_type = pd.DataFrame(auc_t).T[['N'] + models]
df_e6b_type.columns = ['N'] + [ma.MODEL_DISPLAY[m] for m in models]
print(f"\nE6b — TRIANGLE area (title·video·transcript) AUC by fake type   (real N={n_real})")
print("AUC > 0.5 = expected (fake = bigger triangle = less coherent)\n")
display(df_e6b_type)

# Figura
groups = {t: {m: auc_t[t][m] for m in models} for t in GL_TYPES}
fig = ma.grouped_bar(groups, ylabel='AUC(TRIANGLE area)',
                     title='GroundLie360 — TRIANGLE-area detection by fake type (chance = 0.5)',
                     fname='groundlie_e6b_triangle_by_faketype.png', baseline=0.5, pct=False)
plt.show(); plt.close('all')

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score

GL_TYPES = ['false_title', 'false_speech', 'temporal_edit', 'cgi', 'contradictory', 'unsupported']
models = [m for m in GL_MODELS if m in e6b_gl]

def _auc_by_type(df, score_col):
    real = df[df.binary_label == 0]
    out = {}
    for t in GL_TYPES:
        pos = df[df[t] == 1]
        if len(pos) == 0:
            out[t] = np.nan; continue
        y = np.r_[np.ones(len(pos)), np.zeros(len(real))]
        s = np.r_[pos[score_col].values, real[score_col].values]
        out[t] = roc_auc_score(y, s)
    return out

# AUC por tipo: E1 (=-cos título-vídeo) y E6b (=area_true del triángulo)
e1_by, e6_by = {}, {}
for model in models:
    conn = gl_conns[model]
    Ti, Vi = ma.load_globals(conn, 'text_title'), ma.load_globals(conn, 'video')
    df = e6b_gl[model].copy()
    df['e1_negcos'] = [-ma.cos(Ti[r], Vi[r]) for r in df.raw_id]   # alto = más fake (igual convención)
    e1_by[model] = _auc_by_type(df, 'e1_negcos')
    e6_by[model] = _auc_by_type(df, 'area_true')

# ── Figura: dos paneles, mismo eje ────────────────────────────────────────────
with plt.rc_context(ma._RC):
    fig, axes = plt.subplots(1, 2, figsize=(14, 4.4), sharey=True, constrained_layout=True)
    x = np.arange(len(GL_TYPES)); w = 0.8 / len(models)
    panels = [(axes[0], e1_by, 'E1: title \u2194 video  (pairwise cosine)'),
              (axes[1], e6_by, 'E6b: title \u00b7 video \u00b7 transcript  (TRIANGLE area)')]
    for ax, data, ttl in panels:
        for j, model in enumerate(models):
            vals = [data[model][t] for t in GL_TYPES]
            ax.bar(x + j*w - 0.4 + w/2, vals, w,
                   label=ma.MODEL_DISPLAY[model], color=ma.MODEL_COLOR[model])
        ax.axhline(0.5, color='gray', ls='--', lw=1)
        ax.set_xticks(x); ax.set_xticklabels(GL_TYPES, rotation=30, ha='right')
        ax.set_title(ttl); ax.set_ylim(0.2, 0.8)
    axes[0].set_ylabel('AUC  (chance = 0.5)')
    axes[0].legend(fontsize=8, ncol=2, loc='upper left')
    fig.suptitle('GroundLie360 — Pairwise vs Triangle: the triangle recovers CGI', fontsize=13)
    fig.savefig(str(ma.FIG_DIR / 'groundlie_e1_vs_e6b_by_faketype.png'), dpi=150, bbox_inches='tight')
    plt.show(); plt.close('all')

# ── Tabla de ganancia: E6b - E1, por tipo y modelo ────────────────────────────
gain = {t: {ma.MODEL_DISPLAY[m]: round(e6_by[m][t] - e1_by[m][t], 3) for m in models}
        for t in GL_TYPES}
df_gain = pd.DataFrame(gain).T
print("Triangle gain over pairwise (AUC E6b - AUC E1), by fake type:")
display(df_gain)

In [ ]:
from sklearn.metrics import roc_auc_score

GL_TYPES = ['false_title', 'false_speech', 'temporal_edit', 'cgi', 'contradictory', 'unsupported']
models = [m for m in GL_MODELS if m in e6b_gl]

# (a) Co-ocurrencia + casos puros (dataset completo)
ref = gl_labels[models[0]]
fakes = ref[ref.binary_label == 1]
print("=== Co-occurrence (videos with both types) ===")
print(f"{'':14}" + "".join(f"{t[:6]:>8}" for t in GL_TYPES))
for ti in GL_TYPES:
    print(f"{ti:14}" + "".join(f"{int(((fakes[ti]==1)&(fakes[tj]==1)).sum()):>8}" for tj in GL_TYPES))
tsum_full = fakes[GL_TYPES].sum(axis=1)
print("\n=== N total vs PURE (exactly one type) ===")
for t in GL_TYPES:
    print(f"  {t:15} total={int((fakes[t]==1).sum()):4}  pure={int(((fakes[t]==1)&(tsum_full==1)).sum()):4}")
print(f"  reales={int((ref.binary_label==0).sum())}")

# (b) AUC por tipo: triángulo (area_true) y E1 (-cos t-v), marginal vs PURO
e1_neg = {}
for model in models:
    conn = gl_conns[model]
    Ti, Vi = ma.load_globals(conn, 'text_title'), ma.load_globals(conn, 'video')
    e1_neg[model] = {r: -ma.cos(Ti[r], Vi[r]) for r in e6b_gl[model].raw_id}

def _auc(df, score, pos_mask, real_mask):
    p, nn = df[pos_mask], df[real_mask]
    if len(p) == 0: return np.nan
    sp = p[score].values if isinstance(score, str) else np.array([score[r] for r in p.raw_id])
    sn = nn[score].values if isinstance(score, str) else np.array([score[r] for r in nn.raw_id])
    return roc_auc_score(np.r_[np.ones(len(p)), np.zeros(len(nn))], np.r_[sp, sn])

rows = []
for model in models:
    df = e6b_gl[model]; tsum = df[GL_TYPES].sum(axis=1); real_mask = df.binary_label == 0
    for t in GL_TYPES:
        rows.append({'model': ma.MODEL_DISPLAY[model], 'type': t,
                     'N_pure': int(((df[t]==1)&(tsum==1)).sum()),
                     'tri_pure': round(_auc(df, 'area_true', (df[t]==1)&(tsum==1), real_mask), 3),
                     'tri_marg': round(_auc(df, 'area_true', df[t]==1, real_mask), 3),
                     'E1_pure':  round(_auc(df, e1_neg[model], (df[t]==1)&(tsum==1), real_mask), 3)})
df_pure = pd.DataFrame(rows)
print("\n— Triangle area, PURE-type AUC (rows=type, cols=model) —")
display(df_pure.pivot(index='type', columns='model', values='tri_pure'))
print("— Same, MARGINAL (any co-occurring type) —")
display(df_pure.pivot(index='type', columns='model', values='tri_marg'))
print("— E1 (title↔video) PURE-type AUC —")
display(df_pure.pivot(index='type', columns='model', values='E1_pure'))

In [ ]:
print("false_speech — E1:", {ma.MODEL_DISPLAY[m]: round(e1_by[m]['false_speech'],3) for m in models})
print("false_speech — E6b:", {ma.MODEL_DISPLAY[m]: round(e6_by[m]['false_speech'],3) for m in models})
print("N false_speech en E6b:", int((e6b_gl[models[0]]['false_speech']==1).sum()))

In [ ]:
# Close GroundLie360 connections
for conn in gl_conns.values():
    try:
        conn.close()
    except Exception:
        pass
gl_conns.clear()


In [ ]:
# GroundLie360 - AUC + Threshold (NO pareado -> sin 2AFC). Autocontenida.
# fakeness por item (alto = fake): E1 = -cos(video,title); E6b = +area_true(title,video,transcript).
N_REPS_GLR, TRAIN_GLR = 50, 0.7

# --- conexiones y labels PROPIAS (no depende de gl_conns/gl_labels, que la celda 81 cierra) ---
GLR_MODELS = ma.MODELS_BY_DATASET['GroundLie360']
glr_conns, glr_labels = {}, {}
for model in GLR_MODELS:
    if not ma.available('GroundLie360', model).get('exists'):
        continue
    c = ma.connect(ma.db_path('GroundLie360', model))
    try:
        glr_labels[model] = ma.load_labels(c); glr_conns[model] = c
    except Exception as ex:
        print(f"[skip] GLR {model}: {ex}"); c.close()

def _l2(v): n = np.linalg.norm(v); return v / n if n > 1e-12 else v
def _area3(a, b, c):
    a, b, c = _l2(a), _l2(b), _l2(c); u, v = b - a, c - a
    return 0.5 * np.sqrt(max(float(u@u*(v@v) - (u@v)**2), 0.0))
def _fit_thr_high(s, l):
    o = np.argsort(s, kind='mergesort'); ss, ls = s[o], l[o]
    n, nf = len(s), int(l.sum()); cum = np.concatenate([[0], np.cumsum(ls)])
    correct = nf + np.arange(n + 1) - 2 * cum; bk = int(np.argmax(correct))
    if bk == 0: return ss[0] - 1e-9
    if bk == n: return ss[-1] + 1e-9
    return (ss[bk-1] + ss[bk]) / 2
def _thr_balanced(score, label, seed, nrep=N_REPS_GLR, frac=TRAIN_GLR):
    s = np.asarray(score, float); y = np.asarray(label, int)
    pos, neg = np.where(y == 1)[0], np.where(y == 0)[0]; n = min(len(pos), len(neg)); accs = []
    for rep in range(nrep):                                   # subsample balanceado -> chance 0.5
        rng = np.random.default_rng(seed + rep)
        sub = np.concatenate([rng.choice(pos, n, replace=False), rng.choice(neg, n, replace=False)])
        rng.shuffle(sub); k = int(round(frac*len(sub))); tr, te = sub[:k], sub[k:]
        thr = _fit_thr_high(s[tr], y[tr])
        accs.append(((s[te] >= thr).astype(int) == y[te]).mean())
    return float(np.mean(accs))

gl_reg = []
for model in GLR_MODELS:
    if model not in glr_labels: continue
    mods = ma.available('GroundLie360', model).get('modalities', {})
    if not {'text_title', 'video'} <= set(mods):
        print(f"[skip] GL {model}: faltan modalidades"); continue
    conn, lab = glr_conns[model], glr_labels[model]
    Ti = ma.load_globals(conn, 'text_title'); Vi = ma.load_globals(conn, 'video')
    Tr = ma.load_globals(conn, 'transcript') if 'transcript' in mods else {}
    e1_ids = [r for r in set(Ti) & set(Vi) & set(lab.index)]
    f1 = np.array([-ma.cos(Vi[r], Ti[r]) for r in e1_ids]); y1 = np.array([int(lab.loc[r,'binary_label']) for r in e1_ids])
    blocks = [('E1 video-title', f1, y1)]
    if Tr:
        e6_ids = [r for r in set(Ti) & set(Vi) & set(Tr) & set(lab.index)]
        f6 = np.array([_area3(Ti[r], Vi[r], Tr[r]) for r in e6_ids]); y6 = np.array([int(lab.loc[r,'binary_label']) for r in e6_ids])
        blocks.append(('E6b triangle', f6, y6))
    for exp, f, y in blocks:
        auc = float(ma.auc_fake(f, y)); thr = _thr_balanced(f, y, seed=6000)
        gl_reg.append(dict(dataset='GroundLie360', exp=exp, model=model, N=int((y==1).sum()),
                           paired=np.nan, auc=round(auc,3), thr=round(thr,3)))
        print(f"[GL] {ma.MODEL_DISPLAY[model]:13s} {exp:16s} N_fake={int((y==1).sum()):4d}  AUC={auc:.3f}  thr={thr:.3f}")

for c in glr_conns.values():
    try: c.close()
    except Exception: pass

df_gl_reg = pd.DataFrame(gl_reg)
for metric in ['auc', 'thr']:
    print(f"\n== GroundLie {metric} =="); display(df_gl_reg.pivot_table(index='exp', columns='model', values=metric).round(3))


#### 1. Replicar FakeVV: coseno título↔vídeo (E1, AUC)
#### 2. Desglose por tipo: la detectabilidad sigue la MODALIDAD del fake
#### 3. El triángulo título·vídeo·transcripción (E6b)
#### 4. Caveat honesto: el CGI está parcialmente confundido
#### 5. Coherencia entre escenas (E3): el caso del Temporal Edit
#### 6. El principio central (lo más importante)
#### 7. Con qué me quedo


## 6. Dataset C — M3A (gran escala, A1–B)


In [ ]:
M3A_INDEX = REPO / "experiments/M3A/m3a_index_20000.json"
if M3A_INDEX.exists():
    m3a_index = json.load(open(M3A_INDEX, encoding='utf-8'))
    by_id = {e['raw_id']: e for e in m3a_index}
    print(f"M3A index: {len(by_id)} entries")
else:
    print(f"[skip] M3A index not found at {M3A_INDEX}")
    by_id = {}


In [ ]:
M3A_MODELS = ['qw2b', 'qw8b', 'wave', 'ge2']
MOD_ORDER = ['video', 'text_summary', 'transcript',
             'text_nem_person', 'text_nem_location', 'text_nem_organization', 'text_nem_complete']

m3a_inv = {}
for model in M3A_MODELS:
    try:
        conn = ma.connect(ma.db_path('M3A', model))
        rows = dict(conn.execute('SELECT modality, COUNT(*) FROM embeddings GROUP BY modality').fetchall())
        conn.close()
    except Exception as ex:
        print(f"[skip] M3A {model}: {ex}"); rows = {}
    m3a_inv[model] = {m: rows.get(m, 0) for m in MOD_ORDER}        # key por CÓDIGO de modelo

df_m3a_inv = (pd.DataFrame({ma.MODEL_DISPLAY[m]: m3a_inv[m] for m in M3A_MODELS})
                .reindex(MOD_ORDER).replace(0, '—'))
print("M3A — inventario de embeddings por modelo (nº de muestras)")
print("  MM  (mismatch)   → usa video + text_summary + transcript   (los 4 modelos; GE2 subset ~2.5k)")
print("  NEM (entity swap)→ usa text_nem_*                           (solo open-source; GE2 no tiene)")
print("  Modalidades: Texto=summary, Audio=transcript, Vídeo. (Sin imagen, sin MTG/TMG/M2N.)\n")
display(df_m3a_inv)

# Resumen NEM (open-source) y MM (todos)
ref = m3a_inv['qw2b']
print("NEM disponible (open-source), nº por subtipo:")
for sub in ['person', 'location', 'organization', 'complete']:
    print(f"   {sub:13s} {ref[f'text_nem_{sub}']}")
print("\nMM disponible (video+summary+transcript) por modelo:")
for m in M3A_MODELS:
    d = m3a_inv[m]
    print(f"   {ma.MODEL_DISPLAY[m]:12s} video={d['video']}  summary={d['text_summary']}  transcript={d['transcript']}")


In [ ]:
# M3A — setup compartido MM (text-swap): víctimas, helpers y constantes.
# Se define aquí (antes de A1) porque A1 (arista) y A3 (triángulo) lo comparten.
import json
M3A_MM_MODELS = ['qw2b', 'qw8b', 'ge2']   # WAVE fuera: vídeo M3A incompleto (15.294/20.000)

_idx = json.loads((ma.REPO_ROOT / 'experiments/M3A/m3a_index_20000.json').read_text(encoding='utf-8'))
def _parse(v): return eval(v) if isinstance(v, str) and v.startswith('{') else v
mm_text_src = {e['raw_id']: (_parse(e.get('mm_sources')) or {}).get('text')
               for e in _idx if (_parse(e.get('mm_sources')) or {}).get('text')}

def _l2(v): n = np.linalg.norm(v); return v / n if n > 1e-12 else v
def _area3(a, b, c):
    a, b, c = _l2(a), _l2(b), _l2(c); u, v = b - a, c - a
    return 0.5 * np.sqrt(max(float(u@u*(v@v) - (u@v)**2), 0.0))
def _fit_thr_high(s, l):
    o = np.argsort(s, kind='mergesort'); ss, ls = s[o], l[o]
    n, nf = len(s), int(l.sum()); cum = np.concatenate([[0], np.cumsum(ls)])
    correct = nf + np.arange(n + 1) - 2 * cum; bk = int(np.argmax(correct))
    if bk == 0: return ss[0] - 1e-9
    if bk == n: return ss[-1] + 1e-9
    return (ss[bk-1] + ss[bk]) / 2

N_REPS_MM, TRAIN_MM = 50, 0.7

### A1 — Video↔Summary: coherencia real vs MM (text-swap) y control aleatorio


In [ ]:
# A1 — MM (text-swap) con UNA SOLA arista: cos(video, summary). Aísla el enlace que el fake rompe.
# Mismos 3 regímenes (2AFC -> AUC -> threshold). Contrastar con A3 (triángulo): A1 debería igualar o superar.
a1_rows = []
for model in M3A_MM_MODELS:
    conn = ma.connect(ma.db_path('M3A', model))
    S, V = ma.load_globals(conn, 'text_summary'), ma.load_globals(conn, 'video')
    conn.close()
    vics = [i for i, src in mm_text_src.items() if i in S and i in V and src in S]
    if len(vics) < 20:
        print(f"[skip] A1 {model}: solo {len(vics)} pares"); continue
    real = np.array([ma.cos(V[i], S[i])              for i in vics])  # vídeo con SU summary real (alto)
    fake = np.array([ma.cos(V[i], S[mm_text_src[i]]) for i in vics])  # vídeo con summary de otro evento (bajo)
    N = len(vics)

    afc = float((real > fake).mean())                                 # 2AFC: real MÁS alto que el fake
    sc = np.concatenate([-real, -fake]); y = np.array([0]*N + [1]*N)  # -cos -> fake = score alto (conv. auc_fake)
    auc = float(ma.auc_fake(sc, y))                                   # separabilidad global
    grp = np.array(list(range(N))*2); accs = []
    for rep in range(N_REPS_MM):
        rng = np.random.default_rng(2000+rep); perm = rng.permutation(N); k = int(round(TRAIN_MM*N))
        trv = set(perm[:k].tolist()); tr = np.array([g in trv for g in grp]); te = ~tr
        thr = _fit_thr_high(sc[tr], y[tr])                            # corte sobre -cos en train
        accs.append(((sc[te] >= thr).astype(int) == y[te]).mean())
    a1_rows.append({'model': ma.MODEL_DISPLAY[model], 'N_pairs': N,
                    '2AFC': round(afc,3), 'AUC': round(auc,3),
                    'thr_acc': round(float(np.mean(accs)),3), 'thr_std': round(float(np.std(accs)),3)})
    print(f"[A1 cos(V,sum)] {ma.MODEL_DISPLAY[model]:13s} N={N:5d}  2AFC={afc:.3f}  AUC={auc:.3f}  thr={np.mean(accs):.3f}±{np.std(accs):.3f}")

df_m3a_a1 = pd.DataFrame(a1_rows).set_index('model')
print("\nA1 — MM text-swap, una sola arista cos(video, summary). Compara con A3 (triángulo).")
display(df_m3a_a1)

### A3 — Triángulo V·Summary·Transcript


In [ ]:
# A3 — MM text-swap, triángulo summary·vídeo·transcript. Usa el setup compartido (celda anterior).
mm_rows = []
for model in M3A_MM_MODELS:
    conn = ma.connect(ma.db_path('M3A', model))
    S, V, Tr = ma.load_globals(conn, 'text_summary'), ma.load_globals(conn, 'video'), ma.load_globals(conn, 'transcript')
    conn.close()
    vics = [i for i, src in mm_text_src.items() if i in S and i in V and i in Tr and src in S]
    if len(vics) < 20:
        print(f"[skip] MM {model}: solo {len(vics)} pares"); continue
    real = np.array([_area3(S[i], V[i], Tr[i]) for i in vics])               # config real (coherente)
    fake = np.array([_area3(S[mm_text_src[i]], V[i], Tr[i]) for i in vics])   # summary swapped (ot
    N = len(vics)

    # (1) 2AFC pareado: dentro de cada víctima, real < fake
    afc = float((real < fake).mean())

    # (2) AUC/ROC: separabilidad global no pareada, P(area_fake > area_real). Techo de cualquier um
    sc = np.concatenate([real, fake]); y = np.array([0]*N + [1]*N)
    auc = float(ma.auc_fake(sc, y))

    # (3) Threshold: 50 reps 70/30, ajusta corte absoluto en train, evalúa en test (agrupado por víctima)
    grp = np.array(list(range(N))*2)
    accs = []
    for rep in range(N_REPS_MM):
        rng = np.random.default_rng(1000+rep); perm = rng.permutation(N); k = int(round(TRAIN_MM*N))
        trv = set(perm[:k].tolist()); tr = np.array([g in trv for g in grp]); te = ~tr
        thr = _fit_thr_high(sc[tr], y[tr])
        accs.append(((sc[te] >= thr).astype(int) == y[te]).mean())

    mm_rows.append({'model': ma.MODEL_DISPLAY[model], 'N_pairs': N,
                    '2AFC': round(afc, 3), 'AUC': round(auc, 3),
                    'thr_acc': round(float(np.mean(accs)), 3), 'thr_std': round(float(np.std(accs)), 3)})
    print(f"[MM] {ma.MODEL_DISPLAY[model]:13s} N={N:5d}  2AFC={afc:.3f}  AUC={auc:.3f}  thr={np.mean(accs):.3f}±{np.std(accs):.3f}")

df_m3a_mm = pd.DataFrame(mm_rows).set_index('model')
print("\nM3A MM (out-of-context, text-swap) — triángulo summary·vídeo·transcript")
print("Tres regímenes de fácil->difícil:")
print("  2AFC (pareado, cancela baseline) > AUC (separabilidad global, techo de umbral) > threshold (corte absoluto train/test)\n")
display(df_m3a_mm)

### B — NEM (Named Entity Modification) por subtipo


In [ ]:
NEM_SUBS = ['person', 'location', 'organization', 'complete']

def _three_regimes(real, fake, strata, seed):
    """real/fake = cos (alto=real). strata = etiqueta por par; el split 70/30 se hace DENTRO de cada estrato."""
    real = np.asarray(real, float); fake = np.asarray(fake, float); N = len(real)
    afc = float((real > fake).mean())                              # 2AFC: real con cos más alto
    sc = np.concatenate([-real, -fake]); y = np.array([0]*N + [1]*N)
    auc = float(ma.auc_fake(sc, y))                                # separabilidad global
    strata = np.asarray(strata); accs = []
    for rep in range(N_REPS_MM):
        rng = np.random.default_rng(seed + rep); m = np.zeros(N, bool)
        for s in np.unique(strata):                                # 70/30 estratificado por subtipo
            idx = np.where(strata == s)[0]; rng.shuffle(idx)
            m[idx[:int(round(TRAIN_MM*len(idx)))]] = True
        tr = np.concatenate([m, m]); te = ~tr                      # real+fake del par, mismo lado
        thr = _fit_thr_high(sc[tr], y[tr])
        accs.append(((sc[te] >= thr).astype(int) == y[te]).mean())
    return {'N_pairs': N, '2AFC': round(afc,3), 'AUC': round(auc,3),
            'thr_acc': round(float(np.mean(accs)),3), 'thr_std': round(float(np.std(accs)),3)}

b_rows = []
for model in ma.MODELS_BY_DATASET['M3A']:                          # GE2 sin text_nem -> se salta solo
    info = ma.available('M3A', model)
    mods = info.get('modalities', {}) if info.get('exists') else {}
    if not info.get('exists') or 'video' not in mods or 'text_summary' not in mods:
        print(f"[skip] M3A B {model}: video o text_summary ausente"); continue
    conn = ma.connect(ma.db_path('M3A', model))
    V = ma.load_globals(conn, 'video'); T = ma.load_globals(conn, 'text_summary')
    tot_real, tot_fake, tot_strata, any_nem = [], [], [], False
    for si, sub in enumerate(NEM_SUBS):
        Tn = ma.load_globals(conn, f'text_nem_{sub}')
        ids = [r for r in V if r in T and r in Tn]
        if not ids:
            print(f"[skip] B {model} NEM-{sub}: sin datos"); continue
        any_nem = True
        real_ = np.array([ma.cos(V[r], T[r])  for r in ids])       # vídeo con summary real
        fake_ = np.array([ma.cos(V[r], Tn[r]) for r in ids])       # vídeo con entidad cambiada
        res = _three_regimes(real_, fake_, np.zeros(len(ids)), seed=3000 + 100*si)  # subtipo solo
        res.update(model=ma.MODEL_DISPLAY[model], subtype=sub); b_rows.append(res)
        tot_real += real_.tolist(); tot_fake += fake_.tolist(); tot_strata += [sub]*len(ids)
        print(f"[B] {ma.MODEL_DISPLAY[model]:13s} NEM-{sub:12s} N={res['N_pairs']:5d}  "
              f"2AFC={res['2AFC']:.3f}  AUC={res['AUC']:.3f}  thr={res['thr_acc']:.3f}±{res['thr_std']:.3f}")
    if any_nem:                                                    # TOTAL: pool + split estratific
        res = _three_regimes(np.array(tot_real), np.array(tot_fake), tot_strata, seed=3999)
        res.update(model=ma.MODEL_DISPLAY[model], subtype='TOTAL'); b_rows.append(res)
        print(f"[B] {ma.MODEL_DISPLAY[model]:13s} NEM-{'TOTAL':12s} N={res['N_pairs']:5d}  "
              f"2AFC={res['2AFC']:.3f}  AUC={res['AUC']:.3f}  thr={res['thr_acc']:.3f}±{res['thr_std']:.3f}")
    conn.close()

df_b = pd.DataFrame(b_rows)
if not df_b.empty:
    print("\nB — NEM (entity-swap en summary). cos(video, summary). TOTAL con split estratificado por subtipo + desglose.")
    order = ['person','location','organization','complete','TOTAL']
    for metric in ['2AFC', 'AUC', 'thr_acc']:
        print(f"\n== {metric} ==")
        display(df_b.pivot_table(index='model', columns='subtype', values=metric).reindex(columns=order).round(3))

In [ ]:
# --- Plots M3A por experimento: progresión Paired -> AUC -> Threshold ---
D2C = {v: k for k, v in ma.MODEL_DISPLAY.items()}
REG = {'Paired (2AFC)': '2AFC', 'AUC': 'AUC', 'Threshold': 'thr_acc'}   # orden fácil -> difícil

def _by_regime(df):
    return {lab: {D2C[d]: float(df.loc[d, col]) for d in df.index} for lab, col in REG.items()}

def _nem_total_by_regime():
    sl = df_b[df_b['subtype'] == 'TOTAL']
    return {lab: {D2C[m]: float(v) for m, v in zip(sl['model'], sl[col])} for lab, col in REG.items()}

def _nem_by_subtype(metric):
    out = {}
    for sub in ['person', 'location', 'organization', 'complete', 'TOTAL']:
        sl = df_b[df_b['subtype'] == sub]
        out[sub] = {D2C[m]: float(v) for m, v in zip(sl['model'], sl[metric])}
    return out

figs = [
    ma.grouped_bar(_by_regime(df_m3a_a1), 'accuracy / AUC',
                   'M3A — Out-of-context: video vs summary', 'm3a_mm_a1_cos.png', baseline=0.5),
    ma.grouped_bar(_by_regime(df_m3a_mm), 'accuracy / AUC',
                   'M3A — Out-of-context: summary–video–transcript triangle', 'm3a_mm_a3_triangle.png', baseline=0.5),
    ma.grouped_bar(_nem_total_by_regime(), 'accuracy / AUC',
                   'M3A — Entity swap (video vs summary)', 'm3a_nem_total.png', baseline=0.5),
    ma.grouped_bar(_nem_by_subtype('2AFC'), '2AFC accuracy',
                   'M3A — Entity swap: paired accuracy by entity type', 'm3a_nem_2afc.png', baseline=0.5),
]

for f in figs:
    display(f)
    plt.close(f)        # evita que se vuelvan a pintar al cerrar la celda

## 7. Resumen transversal


In [ ]:
# ===== 7. Resumen transversal - ACOPLE de los regímenes ya calculados por sección =====
# Requiere: df_fv_reg (FakeVV), df_gl_reg (GroundLie), df_m3a_a1/df_m3a_mm/df_b (M3A).
D2C = {v: k for k, v in ma.MODEL_DISPLAY.items()}
m3a_reg = []
for exp, df in [('A1 video-summary (MM)', df_m3a_a1), ('A3 triangle (MM)', df_m3a_mm)]:
    for disp in df.index:
        m3a_reg.append(dict(dataset='M3A', exp=exp, model=D2C[disp], N=int(df.loc[disp,'N_pairs']),
                            paired=float(df.loc[disp,'2AFC']), auc=float(df.loc[disp,'AUC']), thr=float(df.loc[disp,'thr_acc'])))
for _, r in df_b[df_b.subtype == 'TOTAL'].iterrows():
    m3a_reg.append(dict(dataset='M3A', exp='B entity-swap (NEM)', model=D2C[r['model']], N=int(r['N_pairs']),
                        paired=float(r['2AFC']), auc=float(r['AUC']), thr=float(r['thr_acc'])))

df_xcut = pd.concat([df_fv_reg, df_gl_reg, pd.DataFrame(m3a_reg)], ignore_index=True)
ORD = {'FakeVV': 0, 'GroundLie360': 1, 'M3A': 2}
df_xcut = df_xcut.assign(_o=df_xcut.dataset.map(ORD)).sort_values(['_o', 'exp']).drop(columns='_o')

for reg, col, src in [('Paired 2AFC', 'paired', df_xcut.dropna(subset=['paired'])),
                      ('AUC (ROC)', 'auc', df_xcut), ('Threshold acc', 'thr', df_xcut)]:
    print(f"\n=== {reg} ===")
    display(src.pivot_table(index=['dataset', 'exp'], columns='model', values=col).round(3))


### Limitaciones honestas


## 8. Experimentos adicionales (Fase 2)


### P1 — Recuperación cross-modal (calidad de alineamiento)


In [ ]:
# P1 — Cross-modal retrieval
_TEXT_MOD = {'FakeVV': 'text_real', 'GroundLie360': 'text_title', 'M3A': 'text_summary'}
p1_rows = []

for dataset, models in ma.MODELS_BY_DATASET.items():
    txt_mod = _TEXT_MOD[dataset]
    for model in models:
        info = ma.available(dataset, model)
        mods = info.get('modalities', {}) if info.get('exists') else {}
        if not info.get('exists') or 'video' not in mods or txt_mod not in mods:
            print(f"[skip] P1 {dataset}/{model}: video or {txt_mod} missing")
            continue
        try:
            conn = ma.connect(ma.db_path(dataset, model))
            V = ma.load_globals(conn, 'video')
            T = ma.load_globals(conn, txt_mod)
            conn.close()
        except Exception as ex:
            print(f"[skip] P1 {dataset}/{model}: load error: {ex}")
            continue

        v2t = ma.retrieval_metrics(V, T, sample_n=1000, seed=0)
        t2v = ma.retrieval_metrics(T, V, sample_n=1000, seed=0)
        for direction, r in [('V2T', v2t), ('T2V', t2v)]:
            p1_rows.append({
                'dataset': dataset, 'model': model, 'direction': direction,
                'n': r['n'],
                'R@1':  round(r['R@1'],  3),
                'R@5':  round(r['R@5'],  3),
                'R@10': round(r['R@10'], 3),
                'MedR': round(r['MedR'], 1),
                'MRR':  round(r['MRR'],  3),
            })
        print(f"  {dataset:12s} {ma.MODEL_DISPLAY[model]:15s}  "
              f"V2T R@1={v2t['R@1']:.3f} MedR={v2t['MedR']:.0f}  |  "
              f"T2V R@1={t2v['R@1']:.3f} MedR={t2v['MedR']:.0f}")

df_p1 = pd.DataFrame(p1_rows) if p1_rows else pd.DataFrame()
if not df_p1.empty:
    display(df_p1.set_index(['dataset', 'model', 'direction']))
else:
    print("[skip] P1: no results collected")


In [ ]:
# P1 — Figure: R@1 (V2T) por modelo y dataset
if not df_p1.empty:
    v2t_df = df_p1[df_p1.direction == 'V2T']
    datasets_p1 = v2t_df['dataset'].unique().tolist()
    r1_data = {}
    for ds in datasets_p1:
        sub = v2t_df[v2t_df.dataset == ds]
        r1_data[ds] = {row['model']: row['R@1'] for _, row in sub.iterrows()}
    fig = ma.grouped_bar(
        r1_data,
        ylabel='R@1 (V2T)',
        title='P1 — Cross-modal retrieval R@1 (Video→Text) by model and dataset',
        fname='p1_retrieval_r1_v2t.png',
        baseline=None,
    )
    plt.show(); plt.close('all')


### P2 — Coherencia como detector: ROC, umbral óptimo y comparación de AUC


In [ ]:
# P2a — GroundLie360 E6b: ROC + AUC[CI] por modelo
p2a_rows = []
p2a_scores_by_model = {}
p2a_labels_by_model = {}

GL_MODELS_P2 = ma.MODELS_BY_DATASET['GroundLie360']
gl_conns_p2 = {}
gl_labels_p2 = {}

for model in GL_MODELS_P2:
    info = ma.available('GroundLie360', model)
    if not info.get('exists'):
        print(f"[skip] P2a {model}: DB not found")
        continue
    conn = ma.connect(ma.db_path('GroundLie360', model))
    try:
        lab = ma.load_labels(conn)
        gl_labels_p2[model] = lab
        gl_conns_p2[model] = conn
    except Exception as ex:
        print(f"[skip] P2a {model}: labels error: {ex}")
        conn.close()

for model in GL_MODELS_P2:
    if model not in gl_labels_p2:
        continue
    info = ma.available('GroundLie360', model)
    mods = info.get('modalities', {})
    needed = ['text_title', 'video', 'transcript']
    missing = [m for m in needed if m not in mods]
    if missing:
        print(f"[skip] P2a {model}: missing {missing}")
        continue
    conn = gl_conns_p2[model]
    lab  = gl_labels_p2[model]
    Ti = ma.load_globals(conn, 'text_title')
    Vi = ma.load_globals(conn, 'video')
    Tr = ma.load_globals(conn, 'transcript')

    recs = []
    for rid in set(Ti) & set(Vi) & set(Tr) & set(lab.index):
        d_tv  = 1 - ma.cos(Ti[rid], Vi[rid])
        d_tt  = 1 - ma.cos(Ti[rid], Tr[rid])
        d_vtr = 1 - ma.cos(Vi[rid], Tr[rid])
        recs.append({
            'raw_id': rid,
            'perim': d_tv + d_tt + d_vtr,
            'binary_label': int(lab.loc[rid, 'binary_label']),
        })
    if not recs:
        print(f"[skip] P2a {model}: no records")
        continue

    df_tmp = pd.DataFrame(recs).set_index('raw_id').sort_index()
    scores = df_tmp.perim.values
    labels_arr = df_tmp.binary_label.values
    # Store id-indexed Series so the paired comparison can align by raw_id
    p2a_scores_by_model[model] = df_tmp.perim
    p2a_labels_by_model[model] = df_tmp.binary_label

    auc, lo, hi = ma.bootstrap_auc_ci(scores, labels_arr, n=2000, seed=0)
    op = ma.roc_operating_point(scores, labels_arr)
    p2a_rows.append({
        'model':  model,
        'n':      len(df_tmp),
        'AUC':    round(auc, 3),
        'CI_lo':  round(lo,  3),
        'CI_hi':  round(hi,  3),
        'thr_youden': round(op['thr_youden'], 4),
        'sensitivity': round(op['sensitivity'], 3),
        'specificity': round(op['specificity'], 3),
    })
    print(f"  P2a {ma.MODEL_DISPLAY[model]:15s}  N={len(df_tmp):4d}  "
          f"AUC={auc:.3f} [{lo:.3f},{hi:.3f}]  "
          f"sens={op['sensitivity']:.3f}  spec={op['specificity']:.3f}")

df_p2a = pd.DataFrame(p2a_rows) if p2a_rows else pd.DataFrame()
if not df_p2a.empty:
    display(df_p2a.set_index('model'))

# Close P2 connections
for conn in gl_conns_p2.values():
    try: conn.close()
    except: pass
gl_conns_p2.clear()


In [ ]:
# P2a — Comparar par de modelos con mayor/menor AUC
if len(p2a_scores_by_model) >= 2:
    models_sorted = sorted(p2a_scores_by_model.keys(),
                           key=lambda m: p2a_rows[[r['model'] for r in p2a_rows].index(m)]['AUC'],
                           reverse=True)
    m_best  = models_sorted[0]
    m_worst = models_sorted[-1]
    # Align both score vectors by raw_id (id-sets differ between models: 1606 vs 1608...).
    # paired_auc_diff_ci requires scores_a/scores_b to refer to the SAME items.
    sa = p2a_scores_by_model[m_best]
    sb = p2a_scores_by_model[m_worst]
    la = p2a_labels_by_model[m_best]
    common_ids = sa.index.intersection(sb.index)
    sa_c = sa.loc[common_ids].values
    sb_c = sb.loc[common_ids].values
    la_c = la.loc[common_ids].values  # labels are identical across models for a given raw_id
    diff, lo_d, hi_d = ma.paired_auc_diff_ci(sa_c, sb_c, la_c, n=2000, seed=0)
    excludes_zero = (lo_d > 0) or (hi_d < 0)
    print(f"Paired AUC diff ({ma.MODEL_DISPLAY[m_best]} - {ma.MODEL_DISPLAY[m_worst]}): "
          f"diff={diff:.3f} CI=[{lo_d:.3f}, {hi_d:.3f}]  n_common={len(common_ids)}  "
          f"CI excludes 0: {excludes_zero}")
else:
    print("[skip] P2a paired comparison: fewer than 2 models with results")


In [ ]:
# P2b — M3A A1: score = -cos(video, summary), real vs MM-text-swap
p2b_rows = []
rng_p2b = np.random.default_rng(42)

for model in ma.MODELS_BY_DATASET['M3A']:
    if not by_id:
        print("[skip] P2b: M3A index not loaded"); break
    info = ma.available('M3A', model)
    mods = info.get('modalities', {}) if info.get('exists') else {}
    if not info.get('exists') or 'video' not in mods or 'text_summary' not in mods:
        print(f"[skip] P2b {model}: video or text_summary missing")
        continue
    try:
        conn = ma.connect(ma.db_path('M3A', model))
        V = ma.load_globals(conn, 'video')
        T = ma.load_globals(conn, 'text_summary')
        conn.close()
    except Exception as ex:
        print(f"[skip] P2b {model}: load error: {ex}")
        continue

    scores_list = []
    labels_list = []
    for rid in V:
        if rid not in T or rid not in by_id:
            continue
        src = by_id[rid].get('mm_sources', {}).get('text')
        if not src or src not in T:
            continue
        # real pair: cos(V, T_real) => score_real = -cos (high => incoherent => fake)
        scores_list.append(-ma.cos(V[rid], T[rid]))
        labels_list.append(0)
        # fake pair: cos(V, T_src)
        scores_list.append(-ma.cos(V[rid], T[src]))
        labels_list.append(1)

    if not scores_list:
        print(f"[skip] P2b {model}: no paired records")
        continue

    scores_arr = np.array(scores_list, float)
    labels_arr = np.array(labels_list, int)
    auc, lo, hi = ma.bootstrap_auc_ci(scores_arr, labels_arr, n=2000, seed=0)
    p2b_rows.append({
        'model': model,
        'n_pairs': len(labels_arr) // 2,
        'AUC': round(auc, 3),
        'CI_lo': round(lo, 3),
        'CI_hi': round(hi, 3),
    })
    print(f"  P2b {ma.MODEL_DISPLAY[model]:15s}  N_pairs={len(labels_arr)//2:5d}  "
          f"AUC={auc:.3f} [{lo:.3f},{hi:.3f}]")

df_p2b = pd.DataFrame(p2b_rows) if p2b_rows else pd.DataFrame()
if not df_p2b.empty:
    display(df_p2b.set_index('model'))


In [ ]:
# P2 — Figure: AUC con barras de error (CI) — GroundLie360 E6b
if not df_p2a.empty:
    with plt.rc_context(ma._RC):
        fig, ax = plt.subplots(figsize=(max(5, 2 * len(df_p2a)), 4.5))
        models_p2a = df_p2a['model'].tolist()
        aucs_p2a   = df_p2a['AUC'].tolist()
        lo_p2a     = df_p2a['AUC'].values - df_p2a['CI_lo'].values
        hi_p2a     = df_p2a['CI_hi'].values - df_p2a['AUC'].values
        colors_p2a = [ma.MODEL_COLOR.get(m, '#888888') for m in models_p2a]
        x_p2a = np.arange(len(models_p2a))
        bars = ax.bar(x_p2a, aucs_p2a, color=colors_p2a, edgecolor='white', linewidth=0.6)
        ax.errorbar(x_p2a, aucs_p2a, yerr=[lo_p2a, hi_p2a],
                    fmt='none', color='black', capsize=5, lw=1.5)
        ax.axhline(0.5, color='#E74C3C', lw=1.4, linestyle='--', alpha=0.85, label='Baseline (0.5)')
        ax.set_xticks(x_p2a)
        ax.set_xticklabels([ma.MODEL_DISPLAY.get(m, m) for m in models_p2a], rotation=15, ha='right')
        ax.set_ylabel('AUC (perimeter score)')
        ax.set_title('P2 — GroundLie360 E6b AUC with 95% Bootstrap CI by model')
        ax.legend(fontsize=8)
        for bar, v in zip(bars, aucs_p2a):
            ax.text(bar.get_x() + bar.get_width()/2, v + 0.005, f'{v:.3f}',
                    ha='center', va='bottom', fontsize=8)
        plt.tight_layout()
        fig.savefig(str(ma.FIG_DIR / 'p2_auc_ci_gl360.png'), dpi=150, bbox_inches='tight')
        plt.show(); plt.close('all')


### P3 — Brecha de modalidad (*modality gap*)


In [ ]:
# P3 — Modality gap (GroundLie360, todos los modelos)
p3_rows = []

for model in ma.MODELS_BY_DATASET['GroundLie360']:
    info = ma.available('GroundLie360', model)
    mods = info.get('modalities', {}) if info.get('exists') else {}
    if not info.get('exists') or 'video' not in mods:
        print(f"[skip] P3 {model}: DB not found or video missing")
        continue
    try:
        conn = ma.connect(ma.db_path('GroundLie360', model))
        Vi = ma.load_globals(conn, 'video')
        pairs_to_test = []
        if 'text_title' in mods:
            Ti = ma.load_globals(conn, 'text_title')
            pairs_to_test.append(('video', 'text_title', Vi, Ti))
        if 'transcript' in mods:
            Tr = ma.load_globals(conn, 'transcript')
            pairs_to_test.append(('video', 'transcript', Vi, Tr))
        conn.close()
    except Exception as ex:
        print(f"[skip] P3 {model}: load error: {ex}")
        continue

    for mod_a, mod_b, A, B in pairs_to_test:
        g = ma.modality_gap(A, B, intra_n=500, seed=0)
        row = {
            'model':     model,
            'pair':      f'{mod_a}↔{mod_b}',
            'n_common':  len(set(A) & set(B)),
            'gap_euclidean':  round(g['gap_euclidean'],  4),
            'mean_inter_cos': round(g['mean_inter_cos'], 4),
            'mean_intra_a':   round(g['mean_intra_a'],   4),
            'mean_intra_b':   round(g['mean_intra_b'],   4),
        }
        p3_rows.append(row)
        print(f"  {ma.MODEL_DISPLAY[model]:15s}  {mod_a}↔{mod_b}  "
              f"gap={g['gap_euclidean']:.4f}  inter_cos={g['mean_inter_cos']:.4f}  "
              f"intra_a={g['mean_intra_a']:.4f}  intra_b={g['mean_intra_b']:.4f}")

df_p3 = pd.DataFrame(p3_rows) if p3_rows else pd.DataFrame()
if not df_p3.empty:
    display(df_p3.set_index(['model', 'pair']))
else:
    print("[skip] P3: no results")


In [ ]:
# P3 — Figure: gap_euclidean por modelo y par de modalidades
if not df_p3.empty:
    pairs_p3 = df_p3['pair'].unique().tolist()
    gap_data_p3 = {}
    for pair in pairs_p3:
        sub = df_p3[df_p3['pair'] == pair]
        gap_data_p3[pair] = {row['model']: row['gap_euclidean'] for _, row in sub.iterrows()}
    fig = ma.grouped_bar(
        gap_data_p3,
        ylabel='Gap euclidean (centroids)',
        title='P3 — Modality gap (Euclidean distance between centroids) by model',
        fname='p3_modality_gap.png',
        baseline=None,
    )
    plt.show(); plt.close('all')


### P4 — Robustez OOD por estrato (geografía y tema)


In [ ]:
# P4 — M3A OOD robustness by geography and topic
if not by_id:
    print("[skip] P4: M3A index not loaded")
else:
    # Choose reference model
    _p4_model = None
    for _cand in ['qw2b', 'qw8b', 'ge2', 'wave']:
        _info = ma.available('M3A', _cand)
        _mods = _info.get('modalities', {}) if _info.get('exists') else {}
        if _info.get('exists') and 'video' in _mods and 'text_summary' in _mods:
            _p4_model = _cand
            break
    if _p4_model is None:
        print("[skip] P4: no M3A model with video+text_summary available")
    else:
        print(f"P4 reference model: {ma.MODEL_DISPLAY[_p4_model]}")
        _conn_p4 = ma.connect(ma.db_path('M3A', _p4_model))
        _V_p4 = ma.load_globals(_conn_p4, 'video')
        _T_p4 = ma.load_globals(_conn_p4, 'text_summary')
        _conn_p4.close()

        # Build scored pairs with stratum labels
        _p4_recs = []
        for _rid in _V_p4:
            if _rid not in _T_p4 or _rid not in by_id:
                continue
            _src = by_id[_rid].get('mm_sources', {}).get('text')
            if not _src or _src not in _T_p4:
                continue
            _geo   = by_id[_rid].get('geography', 'unknown')
            _topic = by_id[_rid].get('topic',     'unknown')
            # real pair
            _p4_recs.append({'score': -ma.cos(_V_p4[_rid], _T_p4[_rid]),
                              'label': 0, 'geography': _geo, 'topic': _topic})
            # fake pair (MM text-swap)
            _p4_recs.append({'score': -ma.cos(_V_p4[_rid], _T_p4[_src]),
                              'label': 1, 'geography': _geo, 'topic': _topic})

        df_p4 = pd.DataFrame(_p4_recs)
        print(f"P4 total scored pairs: {len(df_p4)//2} vídeos")

        # ── By geography ──────────────────────────────────────────────────────
        _geo_rows = []
        for _geo, _grp in df_p4.groupby('geography'):
            _n = _grp.label.eq(0).sum()   # number of real videos in stratum
            if _n < 100:
                continue
            _auc, _lo, _hi = ma.bootstrap_auc_ci(
                _grp.score.values, _grp.label.values, n=2000, seed=0)
            _geo_rows.append({'Geography': _geo, 'N': int(_n),
                               'AUC': round(_auc, 3),
                               'CI_lo': round(_lo, 3), 'CI_hi': round(_hi, 3)})
        df_p4_geo = (pd.DataFrame(_geo_rows)
                       .sort_values('N', ascending=False)
                       .reset_index(drop=True))
        print("\n=== P4 AUC by Geography (N >= 100) ===")
        display(df_p4_geo)

        # ── By topic ──────────────────────────────────────────────────────────
        _topic_rows = []
        for _top, _grp in df_p4.groupby('topic'):
            _n = _grp.label.eq(0).sum()
            if _n < 100:
                continue
            _auc, _lo, _hi = ma.bootstrap_auc_ci(
                _grp.score.values, _grp.label.values, n=2000, seed=0)
            _topic_rows.append({'Topic': _top, 'N': int(_n),
                                 'AUC': round(_auc, 3),
                                 'CI_lo': round(_lo, 3), 'CI_hi': round(_hi, 3)})
        df_p4_topic = (pd.DataFrame(_topic_rows)
                         .sort_values('N', ascending=False)
                         .reset_index(drop=True))
        print("\n=== P4 AUC by Topic (N >= 100) ===")
        display(df_p4_topic)


In [ ]:
# P4 — Figure: horizontal bar chart AUC by geography with error bars
if 'df_p4_geo' in dir() and not df_p4_geo.empty:
    with plt.rc_context(ma._RC):
        _dg = df_p4_geo.sort_values('AUC')  # ascending so best is at top in barh
        _fig_p4, _ax_p4 = plt.subplots(figsize=(8, max(3, 0.45 * len(_dg))))
        _y_pos = np.arange(len(_dg))
        _xerr_lo = (_dg['AUC'] - _dg['CI_lo']).values
        _xerr_hi = (_dg['CI_hi'] - _dg['AUC']).values
        _ax_p4.barh(_y_pos, _dg['AUC'].values,
                    xerr=[_xerr_lo, _xerr_hi],
                    color='#4472C4', alpha=0.75, edgecolor='white',
                    error_kw=dict(ecolor='black', capsize=4, lw=1.2))
        _ax_p4.axvline(0.5, color='#E74C3C', lw=1.4, linestyle='--', alpha=0.85, label='Baseline (0.5)')
        _ax_p4.set_yticks(_y_pos)
        _ax_p4.set_yticklabels(
            [f"{g}  (N={n})" for g, n in zip(_dg['Geography'], _dg['N'])],
            fontsize=8)
        _ax_p4.set_xlabel('AUC (score = −cos(video, summary))')
        _ax_p4.set_title(
            f'P4 — M3A OOD: AUC by Geography ({ma.MODEL_DISPLAY.get(_p4_model, _p4_model)}) '
            f'[strata N≥100, 95% CI]')
        _ax_p4.legend(fontsize=8)
        plt.tight_layout()
        _fig_p4.savefig(str(ma.FIG_DIR / 'p4_m3a_ood_geography.png'), dpi=150, bbox_inches='tight')
        plt.show(); plt.close('all')
        print("Saved p4_m3a_ood_geography.png")
else:
    print("[skip] P4 geography figure: no data")


## 9. Referencias


## 10. Síntesis y cierre — el mapa de lo que el alineamiento semántico detecta


In [ ]:
# ===== 8. Síntesis: comparación emparejada coseno (zero-shot) vs probe (aprendido) =====
# Dos carriles (arista / triángulo), cada uno con sus 2 regímenes (AUC = separabilidad, accuracy = umbral),
# emparejados por modalidades de entrada. Coseno: de df_xcut (E1/A1, E6b/A3). Probe: de probe_results.json.
import json
_PROBE_PATH = ma.REPO_ROOT / "experiments/analysis/probe_results.json"
_probe = json.load(open(_PROBE_PATH, encoding="utf-8")) if _PROBE_PATH.exists() else {}
if not _probe:
    print(f"[nota] {_PROBE_PATH} no encontrado - ejecuta MAVIS_probe_analysis.ipynb primero (df_cmp vacío).")

LANES = {
    "edge":     {"fs": "relational_edge",
                 "exp": {"FakeVV": "E1 video-title", "GroundLie360": "E1 video-title",
                         "M3A": "A1 video-summary (MM)"}},
    "triangle": {"fs": "relational_triangle",
                 "exp": {"GroundLie360": "E6b triangle", "M3A": "A3 triangle (MM)"}},
}

_xc = df_xcut.set_index(["dataset", "exp", "model"])
_rows = []
for lane, cfg in LANES.items():
    for ds, exp in cfg["exp"].items():
        for model, fsd in _probe.get(ds, {}).items():
            if cfg["fs"] not in fsd or (ds, exp, model) not in _xc.index:
                continue
            cos = _xc.loc[(ds, exp, model)]; pr = fsd[cfg["fs"]]
            _rows.append({
                "dataset": ds, "lane": lane, "model": ma.MODEL_DISPLAY.get(model, model),
                "cos_auc": round(float(cos["auc"]), 3), "probe_auc": round(float(pr["auc_mean"]), 3),
                "cos_acc": round(float(cos["thr"]), 3), "probe_acc": round(float(pr["acc_mean"]), 3),
                "N": int(pr["n"]),
            })

df_cmp = pd.DataFrame(_rows)
if not df_cmp.empty:
    df_cmp["d_auc"] = (df_cmp.probe_auc - df_cmp.cos_auc).round(3)
    df_cmp["d_acc"] = (df_cmp.probe_acc - df_cmp.cos_acc).round(3)
    _ORD = {"FakeVV": 0, "GroundLie360": 1, "M3A": 2}
    df_cmp = (df_cmp.assign(_o=df_cmp.dataset.map(_ORD))
                    .sort_values(["lane", "_o", "model"]).drop(columns="_o").reset_index(drop=True))
    print("Comparación emparejada (baseline 0.5). cos_* = coseno zero-shot; probe_* = clasificador lineal.")
    print("AUC = separabilidad (sin umbral); acc = clasificación (umbral train/test); d_* = probe menos coseno.")
    display(df_cmp[["dataset", "lane", "model", "cos_auc", "probe_auc", "d_auc",
                    "cos_acc", "probe_acc", "d_acc", "N"]])
else:
    print("df_cmp vacío.")


In [ ]:
# -- Figura: comparación emparejada coseno vs probe (AUC y accuracy) --
if "df_cmp" in dir() and not df_cmp.empty:
    d = df_cmp.copy()
    d["lbl"] = (d.dataset.replace({"GroundLie360": "GL", "FakeVV": "FVV"})
                + "·" + d.lane.str[:3] + "·"
                + d.model.str.replace("Qwen3-VL ", "Q", regex=False).str.replace("WAVE-7B", "WAVE", regex=False))
    with plt.rc_context(ma._RC):
        fig, axes = plt.subplots(1, 2, figsize=(15, 5))
        for ax, (cc, pc, title) in zip(axes, [("cos_auc", "probe_auc", "AUC — separability"),
                                              ("cos_acc", "probe_acc", "Accuracy — threshold (train/test)")]):
            x = np.arange(len(d)); w = 0.4
            ax.bar(x - w/2, d[cc], w, label="cosine (zero-shot)", color="#9DC3E6",
                   edgecolor="black", hatch="//", alpha=0.85)
            ax.bar(x + w/2, d[pc], w, label="probe (learned)", color="#2E75B6", edgecolor="white")
            ax.axhline(0.5, color="#E74C3C", ls="--", lw=1.2, alpha=0.8)
            ax.set_xticks(x); ax.set_xticklabels(d["lbl"], rotation=75, ha="right", fontsize=7)
            ax.set_ylabel(title.split(" — ")[0]); ax.set_title(title); ax.set_ylim(0.3, 1.0); ax.legend(fontsize=8)
        fig.suptitle("Paired: fixed cosine vs learned probe - per lane (edge/triangle) × dataset × encoder", y=1.02)
        plt.tight_layout()
        fig.savefig(str(ma.FIG_DIR / "synthesis_cos_vs_probe_paired.png"), dpi=150, bbox_inches="tight")
        plt.show(); plt.close("all")
    print("Saved synthesis_cos_vs_probe_paired.png")
else:
    print("[skip] df_cmp vacío - no se genera figura.")


In [ ]:
# -- Síntesis (solo triángulo): coseno (zero-shot) vs probe (aprendido) -> 2 heatmaps (AUC y Accuracy) --
MODELS_ORD = ["GE2", "Qwen3-VL 2B", "Qwen3-VL 8B"]
DS_ORD = ["GroundLie360", "M3A"]                      # FakeVV no tiene triángulo (solo 2 modalidades)

if "df_cmp" in dir() and not df_cmp.empty:
    d = df_cmp[df_cmp.lane == "triangle"].copy()
    row_keys = [ds for ds in DS_ORD if ds in d.dataset.values]
    row_lbl = [ds.replace("GroundLie360", "GroundLie") for ds in row_keys]
    col_lbl, col_pairs = [], []
    for m in MODELS_ORD:
        tag = m.replace("Qwen3-VL ", "Q")
        col_lbl += [f"{tag}\ncos", f"{tag}\nprobe"]
        col_pairs += [(m, "cos"), (m, "probe")]

    idx = d.set_index(["dataset", "model"])
    def _mat(metric):
        M = np.full((len(row_keys), len(col_pairs)), np.nan)
        for i, ds in enumerate(row_keys):
            for j, (m, kind) in enumerate(col_pairs):
                if (ds, m) in idx.index:
                    M[i, j] = idx.loc[(ds, m), f"{kind}_{metric}"]
        return M

    with plt.rc_context(ma._RC):
        fig, axes = plt.subplots(1, 2, figsize=(13, 2.6))
        for ax, (metric, title) in zip(axes, [("auc", "AUC — separability"),
                                              ("acc", "Accuracy — threshold (train/test)")]):
            M = _mat(metric)
            im = ax.imshow(M, cmap="RdYlGn", vmin=0.4, vmax=0.9, aspect="auto")   # rojo -> verde
            ax.set_xticks(range(len(col_lbl))); ax.set_xticklabels(col_lbl, fontsize=8)
            ax.set_yticks(range(len(row_lbl))); ax.set_yticklabels(row_lbl, fontsize=10)
            for i in range(M.shape[0]):
                for j in range(M.shape[1]):
                    if not np.isnan(M[i, j]):
                        ax.text(j, i, f"{M[i, j]:.2f}", ha="center", va="center",
                                fontsize=9, color="black")
            for j in range(2, len(col_pairs), 2):                        # separador entre encoders
                ax.axvline(j - 0.5, color="black", lw=2)
            ax.set_title(title, fontsize=11)
            fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
        fig.suptitle("Triangle coherence — fixed cosine vs learned probe (baseline 0.5)",
                     y=1.04, fontsize=12)
        plt.tight_layout()
        fig.savefig(str(ma.FIG_DIR / "synthesis_triangle_cos_vs_probe.png"), dpi=150, bbox_inches="tight")
        plt.show(); plt.close("all")
    print("Saved synthesis_triangle_cos_vs_probe.png")
else:
    print("[skip] df_cmp vacío.")

In [ ]:
# -- Síntesis (threshold accuracy): heatmap cos | heatmap probe; filas = dataset·carril --
MODELS_ORD = ["GE2", "Qwen3-VL 2B", "Qwen3-VL 8B"]

if "df_cmp" in dir() and not df_cmp.empty:
    d = df_cmp.copy()
    _OD = {"FakeVV": 0, "GroundLie360": 1, "M3A": 2}
    _LO = {"edge": 0, "triangle": 1}
    rows = (d[["dataset", "lane"]].drop_duplicates()
              .assign(_o=lambda x: x.dataset.map(_OD), _l=lambda x: x.lane.map(_LO))
              .sort_values(["_o", "_l"]))
    row_keys = list(zip(rows.dataset, rows.lane))
    row_lbl = [f"{ds.replace('GroundLie360', 'GL').replace('FakeVV', 'FVV')}·{ln}" for ds, ln in row_keys]
    col_lbl = [m.replace("Qwen3-VL ", "Q") for m in MODELS_ORD]

    idx = d.set_index(["dataset", "lane", "model"])
    def _mat(col):
        M = np.full((len(row_keys), len(MODELS_ORD)), np.nan)
        for i, (ds, ln) in enumerate(row_keys):
            for j, m in enumerate(MODELS_ORD):
                if (ds, ln, m) in idx.index:
                    M[i, j] = idx.loc[(ds, ln, m), col]
        return M

    Mc, Mp = _mat("cos_acc"), _mat("probe_acc")
    with plt.rc_context(ma._RC):
        fig, axes = plt.subplots(1, 2, figsize=(9, 2.9), constrained_layout=True)
        for ax, (M, title) in zip(axes, [(Mc, "Cosine (zero-shot)"), (Mp, "Probe (learned)")]):
            im = ax.imshow(M, cmap="RdYlGn", vmin=0.4, vmax=0.9, aspect="auto")   # roj
            ax.set_xticks(range(len(col_lbl))); ax.set_xticklabels(col_lbl, fontsize=9)
            ax.set_yticks(range(len(row_lbl))); ax.set_yticklabels(row_lbl, fontsize=9)
            for i in range(M.shape[0]):
                for j in range(M.shape[1]):
                    if not np.isnan(M[i, j]):
                        ax.text(j, i, f"{M[i, j]:.2f}", ha="center", va="center", fontsize=9, color="black")
            ax.set_title(title, fontsize=11)
        fig.colorbar(im, ax=axes, fraction=0.046, pad=0.04, label="Threshold accuracy")
        fig.suptitle("Threshold classifier accuracy — cosine vs probe (baseline 0.5)", fontsize=12)
        fig.savefig(str(ma.FIG_DIR / "synthesis_threshold_cos_vs_probe.png"), dpi=150, bbox_inches="tight")
        plt.show(); plt.close("all")
    print("Saved synthesis_threshold_cos_vs_probe.png")
else:
    print("[skip] df_cmp vacío.")

### Narrativa de síntesis: cinco movimientos
#### Movimiento 1 — La dificultad del dato
#### Movimiento 2 — Los límites del zero-shot
#### Movimiento 3 — La riqueza del espacio de embedding
#### Movimiento 4 — El techo supervisado (reinterpretado tras la auditoría)
#### Movimiento 5 — El mapa honesto


### Trabajo futuro
